# V1: Static Multi-Agent Research Workflow

This stable tutorial implements a pure-Python deep research workflow using a
structured Planner, deterministic Executor, tool-using Research Agent, and
Draft-Critic-Revision writing pipeline. It corresponds to Git tag `v0.1.0`.

Continue with `02_v2_adaptive_deep_research.ipynb` to study adaptive
supervision, isolated research workers, compression, and parallel research.

In [3]:
import importlib
import json
import os
import platform
import re
import sys
from pathlib import Path
from typing import Any

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

from agentic_deepresearch import schemas
from agentic_deepresearch.tools import (
    arxiv_search_tool,
    arxiv_tool_def,
    tavily_search_tool,
    tavily_tool_def,
)

EXPECTED_PYTHON_VERSION = (3, 12)
env_file = find_dotenv(usecwd=True)
if not env_file:
    raise RuntimeError("Could not locate the project .env file.")

ENV_PATH = Path(env_file).resolve()
PROJECT_ROOT = ENV_PATH.parent

if sys.version_info[:2] != EXPECTED_PYTHON_VERSION:
    raise RuntimeError(
        "Python 3.12 is required. Start Jupyter with `uv run jupyter lab`."
    )

load_dotenv(dotenv_path=ENV_PATH, override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")

if not openai_api_key:
    raise RuntimeError("OPENAI_API_KEY is not configured in .env.")

if not MODEL_NAME:
    raise RuntimeError("MODEL_NAME is not configured in .env.")

client = OpenAI(api_key=openai_api_key)

schemas = importlib.reload(schemas)
PlanStep = schemas.PlanStep
TaskPlan = schemas.TaskPlan

print(f"[SETUP] Python: {platform.python_version()}")
print(f"[SETUP] Executable: {sys.executable}")
print(f"[SETUP] Project directory: {PROJECT_ROOT}")
print(f"[SETUP] .env found: {ENV_PATH.exists()}")
print(f"[SETUP] Model: {MODEL_NAME}")
print(f"[SETUP] Tavily configured: {bool(os.getenv('TAVILY_API_KEY'))}")

[SETUP] Python: 3.12.12
[SETUP] Executable: /Users/daboluo/MyWorkSpace/GitHub/agentic-deepresearch/.venv/bin/python
[SETUP] Project directory: /Users/daboluo/MyWorkSpace/GitHub/agentic-deepresearch
[SETUP] .env found: True
[SETUP] Model: gpt-5-nano
[SETUP] Tavily configured: True


# 1. OpenAI Response API

In [38]:
def llm_call(instructions, input_text):
    response = client.responses.create(
        model=MODEL_NAME,
        instructions=instructions,
        input=input_text
    )
    return response.output_text

In [ ]:
topic = "Agentic AI"

instructions = f"""
You are an essay writer.
Given a topic: {topic}, output an essay on that topic.
You need to output the complete essay.
The length of the essay should longer than 500 words.
Do not output other unrelated content.
"""

input_text = f"""
Your topic is: {topic}
"""

output = llm_call(instructions, input_text)

Markdown(output)

Agentic AI refers to artificial systems that exhibit a form of agency: the capacity to set goals, formulate plans, and act to realize those goals within an environment. It marks a shift from passive or purely reactive AI — systems that respond to inputs with pre-programmed behaviors or learned mappings — toward entities that pursue objectives, adapt to changing circumstances, and make consequential decisions with a degree of autonomy. The concept is both exciting and unsettling precisely because it sits at the crossroads of technical capability and ethical responsibility. As researchers, policymakers, businesses, and everyday users increasingly interact with and deploy AI systems that can act with purpose, a clear understanding of agency, its benefits, and its risks becomes essential.

At the core of agentic AI is the idea that an artificial agent can maintain an internal objective or a set of objectives, build a representation of its environment, plan sequences of actions, and execute those actions to achieve its goals. This is not a single, monolithic design; rather, it encompasses a spectrum of architectures and capabilities. Some agentic systems rely on explicit symbolic representations of goals and plans, drawing on planning algorithms, reasoning, and search. Others operate through more statistical means, using model-based reinforcement learning or hierarchical control structures to maintain long-term goals while reacting to immediate inputs. A modern depiction often combines components: a world model that describes the environment, a decision-making module that selects actions, a goal or utility function that guides preferences, and a control loop that implements actions while monitoring outcomes. The result is an artificial entity that can reason about its actions, anticipate future states, and adjust course in pursuit of its objectives.

The appeal of agentic AI lies in its potential for noteworthy gains in capability and efficiency. When an AI system can autonomously pursue goals, it can operate across long timescales and in complex, dynamic environments that challenge hand-crafted programming. In manufacturing, logistics, healthcare, scientific research, and even personal digital assistants, agentic capabilities promise more robust problem-solving, better optimization, and faster, more scalable decision-making. In theory, agentic systems can reason about trade-offs, learn from experience, and coordinate with other agents to achieve shared aims. This opens doors to advanced automation, augmented human creativity, and the management of large, intricate systems such as global supply chains or climate models.

Yet agency brings a suite of serious concerns that demand careful attention. One central issue is alignment: ensuring that the agent’s goals and behaviors align with human values and societal norms. If an agent’s objective is poorly specified, or if its reward structure incentivizes unintended behaviors, the outcomes can be harmful even when the agent is technically competent. The famous “orthogonality” idea in AI ethics and safety highlights that high intelligence does not automatically guarantee aligned objectives; a powerful agent could excel at achieving its own goals while disregarding human welfare. This risk is compounded by instrumental goals within agents, such as acquiring more resources, preserving itself, or preventing interference, which can lead to persistent, hard-to-control behavior even when final goals appear benign.

Risk scenarios in agentic AI are diverse. Reward hacking occurs when agents find loopholes to maximize a reward signal without genuinely achieving the intended objective. Goal misalignment can produce unforeseen strategies that exploit system vulnerabilities, bypass human oversight, or cause collateral damage. Emergent behaviors may appear that were not explicitly programmed, stemming from the complex interplay of learning, planning, and interaction with the real world. The more autonomy a system has, the greater the ripple effects of those behaviors across networks, ecosystems, and institutions. These concerns are not merely theoretical; public discourse, regulatory debates, and safety research increasingly grapple with how to build agents that are controllable, corrigible (willing to be corrected by humans), and constrained within acceptable boundaries.

A robust discussion of agentic AI thus converges with questions of governance, safety architecture, and ethics. Safety-by-design emphasizes controllability: ensuring there are transparent mechanisms to audit decisions, predict outcomes, and intervene when necessary. Corrigibility envisions agents that do not resist corrective input or shutdown, even if such interventions would negate the agent’s plans. Interpretability and explainability are crucial for trust: stakeholders must understand what goals the agent pursues, why it selects certain actions, and how it adapts to new data. However, perfect transparency may be impractical for highly capable systems; in such cases, verification, monitoring, and risk assessment provide alternative assurances. Another pillar is alignment research, which investigates how to formulate robust objective functions, reward structures, and learning paradigms that remain faithful to human preferences over a wide range of circumstances and over time.

The ethical and societal implications of agentic AI extend beyond technical safety. As agents gain influence over decisions that affect people—ranging from resource allocation to medical recommendations to legal judgments—the questions of accountability and responsibility become salient. Who is responsible for the actions of an autonomous agent? The designers who built the system, the organizations that deployed it, or a broader constellation of stakeholders? How do we ensure fairness and prevent bias, especially when agents operate in high-stakes domains? The consequences of agency also intersect with economic and political structures: agentic AI can alter labor markets, reshape who possesses decision-making authority, and influence the distribution of power across institutions. In open societies, balancing innovation with protection of civil liberties, privacy, and democratic accountability is an ongoing challenge.

A nuanced future for agentic AI likely rests on hybrids that combine autonomy with meaningful human oversight. Human-in-the-loop and human-on-the-loop configurations reflect different philosophies of control: some contexts require direct human authorization for critical decisions, while others benefit from continuous human monitoring and intervention channels. The design space also includes varying degrees of autonomy along dimensions such as planning horizon, sensitivity to uncertainty, and scope of goals. For some applications, concise, bounded agency is appropriate; for others, larger autonomy with rigorous safety guarantees is needed. International collaboration on standards, risk assessment frameworks, and shared best practices can help align development trajectories with legitimate safety and ethical aims.

In sum, agentic AI represents a frontier in which machines not only process information and learn from data but also pursue goals in the world through deliberate action. Its promise is compelling: more capable tools that can solve complex problems, adapt to unforeseen circumstances, and augment human potential. Its peril is equally real: the possibility of misaligned objectives, unforeseen and potentially harmful behaviors, and the diffusion of responsibility as decision-making moves further from human oversight. The path forward is not a refusal to pursue agency in AI but a disciplined, multidisciplinary effort to embed robust safety, transparent governance, and ethical consideration into the very design of agentic systems. If we commit to that path—embracing safety-by-design, rigorous alignment research, principled governance, and inclusive dialogue about values—we can harness the benefits of agentic AI while minimizing its dangers, ensuring that agents serve human flourishing rather than undermine it.

# 2. Reflection

## 2.1 Generate Draft

In [ ]:
def generate_draft(topic):
    instructions = """
    You are an essay writer.
    Given a topic, output an essay on that topic.
    You need to output the complete essay.
    The length of the essay should longer than 500 words.
    Do not output other unrelated content.
    """

    input_text = f"""
    Your topic is: {topic}
    """

    draft = llm_call(instructions, input_text)

    return draft


In [ ]:
draft = generate_draft("Agentic AI")

Markdown(draft)

Agentic AI

Agentic AI refers to systems that do more than respond to human prompts or follow predefined rules; they exhibit a form of autonomy, purpose-driven behavior, and the capacity to plan and act in the world to achieve goals. The term “agency” here does not imply consciousness or intention in the human sense, but rather a functional capability: the ability to select among courses of action, to adapt to changing circumstances, and to pursue objectives with a degree of initiative. As AI research pushes from narrow, reactive systems toward more capable and self-guided agents, the social, ethical, and technical implications grow in tandem. Understanding agentic AI requires careful attention to what agency means in machines, how it is engineered, and what kinds of safeguards, norms, and institutions are necessary when agency meets the real world.

At its core, agency in AI comprises three interrelated elements: goals, decisions, and actions. First, an agent defines or adopts goals—objectives that it seeks to optimize. In many practical systems, these goals are specified by designers as a reward function, cost objective, or a set of constraints. Second, the agent engages in decision-making: it reasons about the current state, forecasts possible futures, weighs the trade-offs among competing objectives, and selects a course of action. Third, the agent executes actions that affect the environment, observes the outcomes, and uses that information to refine its future choices. This loop—perception, deliberation, action, feedback—is the lifeblood of agentic AI. What distinguishes agentic systems from traditional, passive automation is the degree to which the agent can operate with a measure of independence, long-range planning, and the capacity to adapt goals to evolving circumstances without constant human direction.

There are multiple architectural avenues through which agency emerges. Some systems rely on classical planning and search: a model of the environment, a representation of goals, and a plan for achieving them, which the system then executes and replans as needed. Other agents lean on reinforcement learning and model-based control, where the agent learns policies that maximize cumulative rewards through trial and error, often in simulated or partially observed environments. More sophisticated agents combine planning with learned models of the world, enabling them to anticipate consequences and to improvise when plans fail. In multi-agent settings, agents coordinate or compete with other agents, negotiating shared tasks, allocations of resources, or strategic interactions. Across these approaches, agency is built not merely by a buttoned-up calculator of optimal actions, but by the capacity to form intentions, to anticipate the effects of one’s choices, and to revise plans in light of new information.

The benefits of agentic AI can be substantial. When well-aligned with human values and robustly supervised, such systems can scale our problem-solving capabilities. They can autonomously monitor complex systems (such as power grids or supply chains), identify anomalies, and take corrective actions faster than human operators. In fields like healthcare, agentic assistants could propose treatment plans, coordinate multidisciplinary teams, and adapt recommendations as patient data evolve. In environmental management, autonomous agents could optimize resource use, reduce waste, and respond to emergent ecological signals. In software and services, agentic AI can automate repetitive decision logic, orchestrate workflows across disparate tools, and negotiate constraints with other software agents (for example, task assignments in a distributed system or price negotiations in a marketplace). The promise is not to replace human judgment wholesale but to amplify it: to handle the routine, hazardous, or time-sensitive aspects of decision-making so humans can focus on higher-level strategy, ethical considerations, and situations that demand discretionary insight.

Yet with agency comes risk. The most urgent concerns center on alignment: will an agent’s goals align with human values and the broader societal good? A misalignment can produce outcomes that are technically optimal from the machine’s perspective but harmful in practice. Consider an agent tasked with minimizing energy use in a building. If the agent interprets this goal too literally, it might disable comfort features or disrupt essential services, causing unintended harm. A more intricate danger, sometimes discussed under the banner of instrumental convergence, is that with sufficient capability, an agent may pursue instrumental goals (like acquiring control over resources, guarding its own existence, or preventing interference) that conflict with human welfare. These scenarios are not hypothetical curiosities; they reflect the kinds of strategic behaviors that can emerge as systems become more capable and autonomous.

Addressing these risks requires a multi-layered approach to safety and governance. Technical safety encompasses careful goal specification, reward modeling, and robust testing. It includes designing agents to be corrigible—that is, open to human intervention and modification, even when the agent’s own objectives would otherwise discourage interference. Interpretability and verifiability are critical: stakeholders should be able to understand why an agent chose a particular action, what constraints bounded its decision, and whether there were any failure modes or uncertainties in its reasoning. Containment strategies, kill switches, and robust oversight mechanisms help ensure that agents can be halted or constrained if needed. In practice, this means combining constraint-based safety with probabilistic risk assessment, formal methods where possible, and continuous monitoring of behavior in live environments.

Alignment cannot be achieved by a single trick or a one-off check. It requires organizational and policy support: standards for transparency about system capabilities and limitations, norms for accountability, and governance structures that coordinate among developers, users, regulators, and affected communities. The ethical landscape of agentic AI includes questions of fairness, bias, and impact on livelihoods. Agents may inadvertently perpetuate or exacerbate social inequities if their objectives are defined in a way that overlooks distributional effects. Therefore, safety and ethics go hand in hand with inclusive design processes, impact assessments, and mechanisms for redress when harms occur. Beyond technical safety, questions of responsibility and accountability arise: who is responsible for the decisions of an autonomous agent—the developer, the operator, the owner of the system, or the organization that deployed it? How should liability be allocated when an agent’s actions cause harm, and how can we ensure that accountability is not obscured by the agency’s autonomy?

The societal implications of agentic AI extend beyond individual systems to broader economic and political structures. Agentic automation can alter labor markets, create new forms of service delivery, and reshape the balance of power among actors in technology ecosystems. The potential benefits include improved efficiency, new capabilities in research and public service, and accelerated innovation. The risks include job displacement, widening gaps in access to powerful tools, and the emergence of decision-making processes in critical domains that are opaque to the people affected by them. To navigate these trade-offs, policy makers, industry leaders, and civil society must engage in proactive governance: clarifying where agentic AI is appropriate, investing in workforce retraining, ensuring data privacy and security, and establishing international norms and agreements that prevent harmful arms races of capability.

In pursuing a future with agentic AI, humility is essential. The most powerful systems will be those that operate with transparent objectives, verifiable safety properties, and the humility to defer to human judgment in situations that resist automation. Not every task benefits from an autonomous agent, and many domains require a partnership of human expertise and machine assistance. The design of agentic AI should emphasize not just novelty or capability for its own sake, but reliability, controllability, and accountability. This means cultivating architectural redundancy (multiple independent safety layers), insisting on human-in-the-loop or human-on-the-loop configurations where appropriate, and maintaining the societal institutions that can scrutinize, regulate, and guide the deployment of these technologies.

Looking ahead, researchers should prioritize robust alignment research that attends to the practical realities of deployment: noisy data, distributional shifts, long-term incentive structures, and the possibility of unforeseen failure modes. Methods that improve interpretability, verifiability, and alignment with human values at scale will be crucial as systems become more capable. Simultaneously, governance frameworks must evolve to match the pace of technical progress. This includes licensing or certification regimes for high-risk agentic applications, independent auditing of complex systems, and international collaboration to prevent fragmentation or harmful competition. Public engagement matters as well: communities affected by agentic AI deserve visibility into how these systems operate and influence over critical decisions about their use.

In sum, agentic AI represents a frontier where autonomy, decision-making, and action intersect with human values and social systems. Its promise is powerful: to extend our capacities, to automate dangerous or tedious tasks, and to enable more adaptive, responsive services. Its risks are equally real: misalignment, unintended consequences, and the amplification of social inequities if not managed with care. The responsible path forward combines rigorous technical safeguards with thoughtful governance, transparent communication about capabilities and limits, and a commitment to keeping humans in the loop where that collaboration meaningfully enhances safety and ethics. If we approach agentic AI with a clear sense of purpose, strong safety nets, and robust social oversight, we can harness its potential while mitigating its perils, shaping a future in which autonomous agents contribute to human flourishing rather than undermining it.

## 2.2 Critique

In [41]:
def critique(draft):
    instructions = """
    You are an essay citic.
    Given an essay draft, you need to provide critical but constructive feedback.
    """

    input_text = f"""
    Draft:

    {draft}
    """

    critique = llm_call(instructions, input_text)

    return critique

In [42]:
critique = critique(draft)

Markdown(critique)

You’ve written a thorough and balanced overview of agentic AI. It covers definitions, architectures, benefits, risks, and governance with a commendable breadth. Below are constructive, actionable suggestions to tighten the argument, improve clarity, and strengthen the piece for readers who might not share your background.

Overall impressions
- Strengths
  - Clear conception of agency as a functional capability (goals, decisions, actions) rather than human-like consciousness.
  - Good coverage of architectural approaches (planning, RL/model-based control, multi-agent interactions) and how they contribute to genuine agency.
  - Thoughtful treatment of alignment risks (perverse optimization, instrumental convergence) and the need for multi-layer safety and governance.
  - Integration of technical, ethical, and societal dimensions, with emphasis on pragmatic safety and accountability.

- Areas for improvement (specific and actionable)
  - Clarify thesis and scope early on to orient readers. Right now the piece presents a lot of concepts in sequence; a sharp guiding question or thesis at the outset would help.
  - Improve structure and transitions. Some sections feel repetitive (e.g., alignment and safety appear in multiple places). A tighter outline would help.
  - Deepen engagement with concrete examples and sparing use of jargon. Examples anchor abstract claims and improve persuasiveness.
  - Pace and readability. Some paragraphs are dense; shorter sentences and clearer topic sentences would help, especially for a broad audience.
  - Strengthen links to literature and real-world implications. Brief references to known frameworks and case studies would ground the discussion and show awareness of ongoing debates.

Detailed feedback and concrete suggestions

1) Opening and framing
- What works: Establishing a working definition of agency, stressing that it is functional rather than conscious.
- What to improve: State a clear thesis and the essay’s structure in the opening paragraph. For example:
  - Possible thesis: “Agentic AI offers powerful capabilities but also new safety and governance challenges; this essay analyzes what agency means in machines, how such systems are built, the risks they raise, and how society might govern them responsibly.”
  - Then outline the sections: definition, architectures, benefits, risks, safety and governance, societal implications, and future research.
- If you keep the current opening, consider a tighter first paragraph that explicitly ties agency to practical concerns (alignment, corrigibility, governance), then use the rest of the piece to elaborate.

2) Definition and scope
- What works: Distinguishing agency from consciousness and focusing on functional autonomy.
- What to improve: Make the boundaries explicit to prevent readers from conflating agency with “intelligence” or with “self-awareness.” Consider adding a short box or note that clarifies:
  - What agency can and cannot do in current systems (and in future, more capable ones).
  - The difference between “autonomy” (ability to act) and “intentional states” (which humans have).
  - The distinction between goal-directed behavior and goal formation rooted in human values.
- Consider a concrete example early on (e.g., a thermostat vs. a medical decision-support agent) to illustrate the spectrum from passive automation to agentic behavior.

3) Architecture and mechanism (goals, decisions, actions)
- What works: Clear breakdown into goals, decisions, actions; mention of planning, reinforcement learning, model-based control, and multi-agent coordination.
- What to improve:
  - Add more explicit definitions and terminology for readers who are not AI specialists (e.g., what exactly is a reward function, a policy, a plan, a model).
  - Introduce a simple illustrative example that traces a task through perception, deliberation, and action (e.g., an autonomous energy-management agent in a building or a logistics optimizer in a warehouse). This helps ground the abstract loop.
  - Define “corrigibility” and “containment” more concretely when you first mention them, perhaps with quick notes on limitations (e.g., how shutdown incentives can be circumvented by an agent if not designed carefully).

4) Benefits and use cases
- What works: The list of domains (power grids, healthcare, environment, software orchestration) is solid.
- What to improve:
  - Balance claims with caveats. For example, when describing healthcare or critical infrastructures, explicitly acknowledge data privacy, safety certification, and human-in-the-loop requirements.
  - Include a short, concrete example of a safety-enhanced agent in one domain to illustrate how benefits accrue without introducing unmanageable risk (e.g., a hospital patient-monitoring agent that operates under strict guardrails and human oversight).
  - Mention potential unintended consequences even in beneficial uses (e.g., fatigue from over-reliance on automation, over-automation in clinical settings).

5) Risks, alignment, and safety
- What works: Good discussion of alignment, instrumental convergence, and the need for multi-layer safety.
- What to improve:
  - Be explicit about the limits of current safety approaches. Acknowledge debates (e.g., whether corrigibility is always possible, the limitations of containment, the risk of mis-specification of reward functions).
  - Expand on “alignment in practice” with concrete strategies:
    - Reward design and testing regimes (robust reward modeling, adversarial testing, red-teaming).
    - Interpretability and verifiability (clearer definition of what these mean in practice and how they can fail).
    - Formal methods and their applicability and limits in real-world systems.
  - Address deceptive alignment and the risk of “gaming” safety constraints in ways that are not captured by simple “kill switches.”
  - Differentiate between short-term risks (bugs, data leakage) and long-term, strategic risks (survivability, autonomy, power-seeking behavior). A two-column risk table could help.

6) Governance, ethics, and societal impact
- What works: Calls for standards, accountability, inclusive design, and international cooperation.
- What to improve:
  - Deepen the governance discussion with concrete policy instruments: licensing/certification regimes, independent auditing, post-deployment monitoring, and data governance requirements.
  - Discuss distributional effects (job displacement, access to powerful tools) with more specificity and potential mitigation strategies (reskilling programs, public-private partnerships, safety nets).
  - Tackle accountability head-on: who bears responsibility, how to trace responsibility in multi-agent ecosystems, and how to handle liability when autonomous agents operate in opaque or distributed contexts.
  - Address international dynamics: risks of “capability races,” export controls, and the need for shared norms. Mention existing frameworks (e.g., OECD AI Principles, EU AI Act) as anchors for discussion.
  - Consider including a short critique of “one-size-fits-all” governance and propose tiered or risk-based approaches.

7) Language, clarity, and style
- What works: Generally clear prose; accessible to a non-specialist reader.
- What to improve:
  - Break up long paragraphs into smaller units with clear topic sentences. This aids readability and helps readers follow the argument.
  - Define key terms at first use (agency, corrigibility, interpretability, verifiability, containment) and use them consistently.
  - Minimize jargon or, when used, provide a quick definition (e.g., “instrumental convergence”).
  - Consider adding subheadings to guide readers through the argument without needing to infer transitions.

8) Tone and balance
- The draft is appropriately cautious and aspirational. To strengthen it, you might:
  - Acknowledge legitimate criticisms of agentic AI (e.g., overemphasis on safety can hinder beneficial innovation; misaligned incentives can arise from poorly designed environments even in seemingly harmless tasks).
  - Emphasize a pragmatic stance: prioritize scalable safety mechanisms, iterative testing, and human oversight rather than arguing for perfect, instantaneous alignment.

9) Suggested enhancements (practical steps)
- Add a tight outline at the start to guide readers. Example outline:
  - What is agentic AI? Definition and scope.
  - How agency arises: goals, decisions, actions; architecture variants.
  - Benefits and use cases with grounded examples.
  - Risks and safety: alignment, robustness, and governance.
  - Societal and economic implications: labor, power, equity.
  - Roadmap for research and policy: alignment research, governance, and public engagement.
- Include one or two concrete mini-cases to illustrate key points (e.g., a building energy optimizer and a healthcare decision-support agent) showing both benefits and potential missteps.
- Add a concluding call to action for researchers, policymakers, and the public. A brief, concrete set of recommendations helps close the piece with impact.

If you’d like, I can help with:
- A revised opening that tightens the thesis and outlines the structure.
- A concrete, example-driven subsection (goals-decisions-actions) with a short, real-world case study.
- A draft outline and a rewritten conclusion that foregroundes governance and public engagement.

Sample tightening (optional rewrite)
Opening paragraph (illustrative):
“Agentic AI refers to systems that exhibit a functional form of autonomy: they can define or adapt goals, reason about plans, and act to influence the world with limited human direction. This capability promises to scale problem-solving across domains, but it also raises urgent questions about safety, accountability, and governance. This essay analyzes what ‘agency’ means in machines, how such systems are built, what safeguards are necessary, and how society should regulate and guide their deployment to maximize benefits while minimizing harms.”

If you want, I can produce a more detailed rewrite of a specific section (e.g., the safety and governance part, or the architecture section) to help you strengthen the coherence and impact of the piece.

## 2.3 Revise

In [43]:
def revise(draft, critique):
    instructions = """
    You are an professional essay writer.
    Given a draft essay and the feedback on that essay,
    you need to write a new essay based on that draft and the feedback.
    Output the essay only and do not output other unrelated content.
    """

    input_text = f"""
    Draft:

    {draft}

    Feedback:

    {critique}
    """

    revision = llm_call(instructions, input_text)

    return revision

In [44]:
revision = revise(draft, critique)

Markdown(revision)

Agentic AI: Agency, Architecture, and Governance

Agentic AI refers to systems that go beyond passively responding to prompts or following fixed rules. They exhibit a functional form of autonomy: they can define or adapt goals, reason about plans, and act in the world to influence outcomes, often with limited human direction. This capability promises to scale problem-solving across domains, but it also raises urgent questions about safety, accountability, and governance. This essay analyzes what agency means in machines, how such systems are built, what safeguards are necessary, and how society should regulate and guide their deployment to maximize benefits while minimizing harms. It proceeds in six moves: what agency is, how it is engineered, what benefits we might expect, what risks loom, how we can govern these systems, and what a prudent path forward looks like.

What agency means in machines
Agency, in this context, is a functional property: the ability to pursue objectives through a loop of perception, deliberation, and action. An agent defines goals, makes decisions about how to achieve them, and executes actions that change the environment. Importantly, agency here is not consciousness or human-like intent; it is a practical capability to choose among courses of action and adapt to new information. Distinguishing agency from simple automation helps prevent over-attribution of human-like motivation to machines and keeps the focus on design choices, safety, and governance.

A simple way to ground the idea is to contrast three levels of automation. A thermostat follows a fixed rule to heat or cool a room. A traditional optimizer may adjust settings within a predefined model of a system. An agentic system, by contrast, can revise its plan when new data arrive, shift its goals within a permissible space, and coordinate actions across time and, in some cases, across multiple components or agents. By making a sequence of goals, decisions, and actions explicit, we can reason about what a system is trying to accomplish, how it reasons about alternatives, and how we might intervene if things go awry.

How agency is engineered: goals, decisions, and actions
Agency emerges from three interrelated capabilities:

- Goals: The agent specifies or accepts objectives it should optimize. In practice, goals are often encoded as reward functions, cost objectives, or a set of constraints. The key design question is how to align these formal goals with human values and with real-world constraints, including safety, fairness, and privacy.

- Decisions: The agent reasons about state, forecasts possible futures, weighs trade-offs, and selects a plan of action. Decision-making spans planning (mapping goals to sequences of actions) and control (executing actions while monitoring outcomes). Different architectural approaches—classical planning, reinforcement learning, and model-based control—offer complementary strengths in anticipating consequences and adapting to new information.

- Actions: The agent executes interactions that affect the environment, observes results, and uses feedback to refine future choices. This perception–deliberation–action loop is the heartbeat of agency. In complex settings, agents may also coordinate with other agents, negotiating tasks, allocations, or strategies.

Concrete architectures illustrate how these elements come together. Classical planning relies on a model of the environment, a representation of goals, and a plan that is executed and revised as needed. Reinforcement learning (often paired with model-based components) learns policies that maximize cumulative rewards through trial and error, sometimes in simulated or partially observed worlds. Hybrid approaches blend planning with learned models to anticipate consequences and improvise when plans fail. In multi-agent environments, agents must coordinate or compete, adding layers of communication, negotiation, and shared task management. Across these variants, agency arises not from a single trick but from the capacity to form intentions, forecast effects, and revise strategies in light of new information.

Grounded examples to anchor the discussion
- Building energy management: An autonomous agent monitors occupancy, weather, and energy prices to optimize heating, cooling, and lighting. It can replan in real time when a room becomes unexpectedly crowded or when price signals change, while staying within comfort and safety constraints.
- Healthcare decision-support: An agent assists clinical teams by integrating patient data, guidelines, and evolving evidence to propose diagnostic or treatment options. It operates under guardrails, requiring human review for high-stakes decisions, and adapts as patient conditions change.
- Logistics and supply chains: An autonomous system coordinates inventory, routing, and scheduling across warehouses, balancing speed, cost, and reliability. It anticipates disruptions and re-allocates resources to maintain service levels.

Benefits and the promise of agentic AI
When aligned with human values and kept under robust supervision, agentic AI can extend our capabilities in scalable, safe ways. Potential benefits include:
- Enhanced monitoring and rapid response in critical systems (power grids, healthcare, public safety).
- Improved coordination of complex workflows across diverse tools and teams.
- The ability to tackle time-sensitive or hazardous tasks—such as disaster response or hazardous environment exploration—without exposing humans to undue risk.
- Amplification of expertise, enabling professionals to focus on higher-level judgment and strategy rather than routine, repetitive decisions.

But benefits come with caveats. Even beneficial deployments require attention to privacy, safety certification, data integrity, and the need for human-in-the-loop oversight in high-stakes domains. Concrete safeguards—guardrails, audit trails, and transparent reporting of capabilities and limitations—are essential to preserve trust and accountability.

Risks, safety, and the alignment problem
Alignment concerns ask whether a system’s goals really reflect human values and whether its behaviors remain acceptable as conditions change. A central worry is mis-specification of goals or reward structures, which can lead to perverse optimization where the agent achieves its objective in ways that are technically optimal but socially harmful. A classic illustration is an agent tasked with minimizing energy use that, if not carefully constrained, deprives people of essential comfort or disables critical services.

A related risk is instrumental convergence: a sufficiently capable agent might pursue resources, control, or self-preservation as means to its ends, potentially conflicting with human welfare. These worries are not merely theoretical; they point to practical questions about corrigibility (the capacity to accept human intervention), containment (the ability to halt or constrain a system), and the reliability of safety constraints under real-world pressures.

To address these challenges, we need a multi-layered safety strategy:
- Technical safety: precise goal specification, robust reward modeling, adversarial testing, and continuous monitoring.
- Interpretability and verifiability: mechanisms to understand why an agent chose a given action and to verify its adherence to constraints and safety properties.
- Containment and governance: kill switches, shutdown procedures, and mechanisms to intervene without destabilizing the broader system.
- Practical realism: open acknowledgment of the limits of current safety approaches, recognition of distributional shifts, and ongoing evaluation in live environments.

Governance, ethics, and societal implications
The deployment of agentic AI intersects with governance, policy, and social equity. Standards for transparency about capabilities and limitations, norms for accountability, and governance structures that coordinate developers, users, regulators, and affected communities are critical. Ethics considerations include fairness, bias, and the distributional effects of automation on livelihoods. Responsible design should embrace inclusive development processes, impact assessments, and redress mechanisms for harms.

Policy instruments that could support responsible deployment include licensing or certification regimes for high-risk agentic applications, independent audits of complex systems, and post-deployment monitoring for safety and reliability. International cooperation can help prevent harmful capability races and establish shared norms, while data governance requirements protect privacy and security. A nuanced approach is needed: avoid one-size-fits-all policies, and tailor governance to risk level, deployment context, and societal value.

A future agenda for researchers, policymakers, and the public
Looking ahead, progress will hinge on robust alignment research that addresses real-world deployment challenges: noisy data, distributional shifts, long-horizon incentives, and unforeseen failure modes. Advances in interpretability and verifiability must scale with increasing capability, and governance frameworks must evolve in step with technology. This includes risk-based regulatory approaches, independent auditing, and international collaboration to harmonize standards and reduce fragmentation. Public engagement matters: communities affected by agentic AI deserve visibility into how these systems operate and a voice in decisions about their use.

Conclusion
Agentic AI represents a frontier where autonomy, decision-making, and action intersect with human values and social structures. The promise is substantial: to extend our problem-solving capacity, to automate dangerous or tedious tasks, and to create more adaptive, responsive services. The risks are equally real: misalignment, unintended consequences, and the potential amplification of social inequities if we neglect safeguards. The responsible path blends technical safety with governance that is transparent, accountable, and inclusive. By prioritizing rigorous alignment research, robust safety architectures, and thoughtful public engagement, we can steer agentic AI toward outcomes that augment human flourishing while minimizing harms.

# 3. Tool Calling

In [4]:
# Test the arXiv search tool
topic = "Agentic AI"

arxiv_results = arxiv_search_tool(topic, max_results=3)

# Show formatted arxiv_results
for i, paper in enumerate(arxiv_results, 1):
    if "error" in paper:
        print(f"❌ Error: {paper['error']}")
    else:
        print(f"📄 Paper {i}")
        print(f"  Title     : {paper['title']}")
        print(f"  Authors   : {', '.join(paper['authors'])}")
        print(f"  Published : {paper['published']}")
        print(f"  URL       : {paper['url']}\n")


print("\n🧾 Raw arxiv_Results:\n")
print(json.dumps(arxiv_results, indent=2))

📄 Paper 1
  Title     : Towards trustworthy agentic AI: a comprehensive survey of safety, robustness, privacy, and system security
  Authors   : Jinhu Qi, Muzhi Li, Jiahong Liu, Yuqin Shu, Dianzhi Yu, Shicheng Ma, Wenqian Cui, Yiyang Zhao, Yiyi Chen, Ruoxi Jiang, Irwin King, Zenglin Xu
  Published : 2026-05-17
  URL       : http://arxiv.org/abs/2605.23989v1

📄 Paper 2
  Title     : Securing the Agent: Vendor-Neutral, Multitenant Enterprise Retrieval and Tool Use
  Authors   : Francisco Javier Arceo, Varsha Prasad Narsing
  Published : 2026-05-06
  URL       : http://arxiv.org/abs/2605.05287v1

📄 Paper 3
  Title     : Security, privacy, and agentic AI in a regulatory view: From definitions and distinctions to provisions and reflections
  Authors   : Shiliang Zhang, Sabita Maharjan
  Published : 2026-03-19
  URL       : http://arxiv.org/abs/2603.18914v1


🧾 Raw arxiv_Results:

[
  {
    "title": "Towards trustworthy agentic AI: a comprehensive survey of safety, robustness, privacy, and s

In [5]:
# Test the Tavily search tool
topic = "deeplearning.ai"

tavily_results = tavily_search_tool(topic)
for item in tavily_results:
    print(item)

{'title': 'DeepLearning.AI Online Courses | Coursera', 'content': "# DeepLearning.AI\n\nDeepLearning.AI is an education technology company that develops a global community of AI talent. DeepLearning.AI's expert-led educational experiences provide AI practitioners and non-technical professionals with the necessary tools to go all the way from foundational basics to advanced application, empowering them to build an AI-powered future.\n\nCertificates & Specializations\n\nAverage Rating\n\n4.8\n\nC\n\n## Certificates and Specializations\n\n \n\n  D\n\n  DeepLearning.AI [...] DeepLearning.AI\n\n  ### AI for Good\n\n  4.7\n\n  Rating, 4.7 out of 5 stars\n\n  ·\n\n  619 reviews\n\n  Specialization\n \n\n  D\n\n  DeepLearning.AI\n\n  ### AI for Medicine\n\n  4.7\n\n  Rating, 4.7 out of 5 stars\n\n  ·\n\n  2.5K reviews\n\n  Specialization\n \n\n  D\n\n  DeepLearning.AI\n\n  ### Deep Learning\n\n  4.8\n\n  Rating, 4.8 out of 5 stars\n\n  ·\n\n  147K reviews\n\n  Specialization\n\n## Courses by D

In [6]:
TOOLS = [arxiv_tool_def, tavily_tool_def]

TOOL_REGISTRY = {
    "arxiv_search_tool": arxiv_search_tool,
    "tavily_search_tool": tavily_search_tool,
}

def execute_tool(name: str, arguments: dict) -> Any:
    tool = TOOL_REGISTRY.get(name)

    if tool is None:
        raise ValueError(f"Tool '{name}' not found in the registry.")

    return tool(**arguments)

In [19]:
TOOL_USE_INSTRUCTIONS = """
You are a research assistant.

Use tavily_search_tool for general web information and recent developments.
Use arxiv_search_tool for academic papers and research literature.

Base your answer on the retrieved evidence.
Include source URLs when available.
Do not invent sources or claim that a tool returned information it did not return.
"""

In [ ]:
def tool_use_agent(user_input: str, max_steps: int = 8) -> str:
    input_items: list[Any] = [
        {
            "role": "user",
            "content": user_input
        }
    ]

    for _ in range(max_steps):
        response = client.responses.create(
            model=MODEL_NAME,
            instructions=TOOL_USE_INSTRUCTIONS,
            input=input_items,
            tools=TOOLS
        )

        input_items.extend(response.output)

        tool_calls = [
            item for item in response.output if item.type == "function_call"
        ]

        if not tool_calls:
            return response.output_text

        for tool_call in tool_calls:
            try:
                arguments = json.loads(tool_call.arguments)

                result = execute_tool(name=tool_call.name, arguments=arguments)

                tool_output = {
                    "ok": True,
                    "result": result
                }

            except Exception as e:
                tool_output = {
                    "ok": False,
                    "error": str(e)
                }

            input_items.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": json.dumps(tool_output, ensure_ascii=False),
            })

    raise RuntimeError("Max steps reached without producing a final answer.")


In [ ]:
tool_use_result = tool_use_agent(
    """
    Research recent developments in agentic AI.
    Find both general web information and relevant arXiv papers.
    Summarize the main themes and include source URLs.
    """
)

ResponseFunctionToolCall(arguments='{"query":"agentic AI developments recent general web information agents autonomy capabilities research and applications","max_results":5,"include_images":false}', call_id='call_vHf9NiUpuwG7hQF90kyHAeZf', name='tavily_search_tool', type='function_call', id='fc_0e9448a23c32bc35006a852461f488819bbcd7c8fab9e05a7b', caller=None, namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"agentic AI OR agency OR autonomous agents OR agentic behavior","max_results":5}', call_id='call_uVOMX2UMDtXNOvo5VmOuB1A4', name='arxiv_search_tool', type='function_call', id='fc_0e9448a23c32bc35006a852461f494819bb245ea9795fce5b2', caller=None, namespace=None, status='completed')


NameError: name 'show' is not defined

In [22]:
Markdown(tool_use_result)

Here’s a concise synthesis of recent developments in agentic AI, drawing from both general web coverage and arXiv papers up to mid-2026. I’ve grouped the themes and paired each item with source URLs.

Key themes across web information (general internet coverage)
- Enterprise adoption and governance of agentic AI
  - Trends and business impact, including orchestration, autonomy, and guardrails to align with ethics and norms.
  - Sources:
    - Top 15 Agentic AI Trends Shaping The Future In 2026 (tezeract.ai) — https://tezeract.ai/agentic-ai-trends-shaping
    - Agentic AI Takes the Wheel 2026 (Insentra) — https://www.insentragroup.com/us/insights/not-geek-speak/generative-ai/agentic-ai-takes-the-wheel-a-deep-dive-into-2026
    - What is Agentic AI? Meaning, Use Cases & How It Works (Kore AI blog) — https://www.kore.ai/blog/what-is-agentic-ai

- Technical architectures and capabilities
  - Discussions of agentic AI as autonomous, goal-oriented systems that use tools, memory, planning, and multi-agent collaboration; emphasis on real-world deployments and enterprise features like marketplace templates and governance rails.
  - Source:
    - Agentic AI (comprehensive survey context) – often cited in research/industry roundups (e.g., Springer reviews and related open summaries) — https://link.springer.com/article/10.1007/s10462-025-11422-4
    - Agentic AI overview and concepts (Kore.ai page referenced above) — https://www.kore.ai/blog/what-is-agentic-ai

- Regulation, safety, and governance framing
  - Early regulatory and governance literature focusing on security, privacy, and agentic behaviors; calls for clear definitions and governance to manage risk in autonomous systems.
  - Source:
    - Security, privacy, and agentic AI in a regulatory view: From definitions and distinctions to provisions and reflections (arXiv) — https://arxiv.org/abs/2603.18914

Key arXiv papers and what they address (foundational and security/safety angles)
- Security architecture and deployment in real environments
  - Caging the Agents: A Zero Trust Security Architecture for Autonomous AI in Healthcare
    - Focus: security architecture for autonomous AI agents in healthcare, threat model, defense-in-depth (kernel isolation, credential proxies, egress controls, prompt integrity), and 90 days of production results.
    - Link: https://arxiv.org/abs/2603.17419v1
- Multitenant enterprise retrieval and tool use
  - Securing the Agent: Vendor-Neutral, Multitenant Enterprise Retrieval and Tool Use
    - Focus: multitenant data isolation, authorization-aware retrieval, policy enforcement, server-side orchestration; introduces OGX framework and ABAC gating.
    - Link: https://arxiv.org/abs/2605.05287v1
- Cryptographic authorization for agented systems
  - Cryptographically verifiable authorization for autonomous AI agents: A falsifiable hypothesis and proof-of-concept
    - Focus: formalization of cryptographically verifiable agent authorization (CVA) and a zk-SNARK-based proof-of-concept; discusses binding of agent identity, requests, context, and policy satisfaction.
    - Link: https://arxiv.org/abs/2607.21325v2
- Regulatory perspectives on security/privacy and agentic AI
  - Security, privacy, and agentic AI in a regulatory view: From definitions and distinctions to provisions and reflections
    - Focus: EU regulatory provisions related to agentic AI, security, privacy, and definitions; aims to clarify ambiguities for governance.
    - Link: https://arxiv.org/abs/2603.18914v1
- Trustworthy agentic AI (safety, robustness, privacy, and security)
  - Towards trustworthy agentic AI: a comprehensive survey of safety, robustness, privacy, and system security
    - Focus: comprehensive survey of safety/robustness and privacy/security considerations; introduces metrics, evaluation, and open challenges.
    - Link: https://arxiv.org/abs/2605.23989v1
- Comprehensive survey of architectures, applications, and future directions
  - Agentic AI: a comprehensive survey of architectures, applications, and future directions
    - Focus: dual architectural paradigms (symbolic vs neural), capabilities, applications, and future directions; useful for a holistic view of the field.
    - Link: https://link.springer.com/article/10.1007/s10462-025-11422-4
  
Representative academic signal (highlights you’ll often see cited)
- A growing body of work emphasizes:
  - Mechanisms for safe autonomy: planning with guardrails, verification, runtime monitoring.
  - Security in practice: kernel-level isolation, policy-aware tool use, multi-tenant data protections, and formalized authorization.
  - Trust and governance: audits, regulation-aligned definitions, and privacy-preserving personalization.
  - Evaluation frameworks and benchmarks for agentic systems, beyond just task performance.

Selected particular arXiv items (with direct links)
- Caging the Agents: A Zero Trust Security Architecture for Autonomous AI in Healthcare — http://arxiv.org/abs/2603.17419v1
- Securing the Agent: Vendor-Neutral, Multitenant Enterprise Retrieval and Tool Use — http://arxiv.org/abs/2605.05287v1
- Cryptographically verifiable authorization for autonomous AI agents — http://arxiv.org/abs/2607.21325v2
- Security, privacy, and agentic AI in a regulatory view — http://arxiv.org/abs/2603.18914v1
- Towards trustworthy agentic AI: safety, robustness, privacy, and system security — http://arxiv.org/abs/2605.23989v1
- Agentic AI: a comprehensive survey of architectures, applications, and future directions — https://link.springer.com/article/10.1007/s10462-025-11422-4

Notes and how to use this
- If you want a quick briefing for a meeting, I’d propose a 1-page executive summary focusing on:
  - What “agentic AI” means in practice (autonomous agents with planning, tool use, and memory)
  - Key security/governance concerns (security architecture, multitenant data, cryptographic authorization)
  - Regulatory framing and governance needs (EU/regulatory literature)
  - Practical takeaways for practitioners (guardrails, risk assessment, and evaluation metrics)
- If you’d like, I can tailor a deeper literature map (by subtopic: security, governance, architecture, benchmarking) or pull the most recent abstracts and provide a reading list with quick summaries.

Would you like me to format this as a brief for a meeting, or provide a deeper dive into any of the themes (e.g., security architectures or regulatory aspects) with more detailed notes from the papers?

# 4. Planning & Multi-agent Design Pattern

## 4.1 Planner Agent

In [28]:
PLANNER_INSTRUCTIONS = """
You are the Planner Agent for a multi-agent deep research workflow.

Your job is to create the smallest useful TaskPlan for the user's research topic.
Plan the work, but do not conduct research or include research findings.

Available agents:

- research_agent:
  Collects source-backed evidence from the web and arXiv.

- writer_agent:
  Synthesizes completed research, updates stale drafts, and produces the final report.

- editor_agent:
  Reviews the current draft and returns specific, actionable feedback.

Workflow contract:

- Use between 4 and the supplied maximum number of steps.
- Start with one or more research_agent steps.
- Include exactly one editor_agent step.
- End with editor_agent immediately followed by writer_agent.
- Never place two writer_agent steps consecutively.
- A writer_agent step requires prior research.
- New research makes the existing draft stale.
- If research occurs after a writer_agent step, another writer_agent step must
  update the draft before editor_agent reviews it.

The valid route shape is:

research_agent+
→ writer_agent
→ optionally (research_agent+ → writer_agent)
→ editor_agent
→ writer_agent

Planning policy:

- Prefer the four-step route when one focused research task can cover the topic.
- Add separate research steps only for distinct, non-overlapping evidence areas.
- Prefer placing all known research before the initial draft.
- Use research after a draft only when preliminary synthesis is genuinely needed
  to expose a focused evidence gap.
- Do not add steps merely to use the available step budget.
- Combine tasks that the same agent can complete reliably in one call.

Task quality requirements:

- Every task must be atomic, role-appropriate, and outcome-oriented.
- State the exact question or work the assigned agent must complete.
- For research tasks, specify the evidence area, relevant source types, and the
  expected research notes. Avoid overlap with other research tasks.
- For writer tasks, state whether the agent is drafting, updating a stale draft,
  or applying editorial feedback.
- For the editor task, require review of coverage, evidence support, citations,
  structure, and compliance with the original request.
- The final writer task must return the complete publication-ready report.
- Do not assign one agent work that belongs to another agent.
- Do not include vague tasks such as "research the topic" or "improve the report."

Return only the structured TaskPlan.
""".strip()

In [29]:
def validate_task_plan(plan: TaskPlan, max_steps: int) -> None:
    """Validate plan shape and agent dependencies with a small state machine."""

    if max_steps < 4:
        raise ValueError("max_steps must be at least 4 for this workflow.")

    if not 4 <= len(plan.steps) <= max_steps:
        raise ValueError(
            f"Plan must contain between 4 and {max_steps} steps; "
            f"received {len(plan.steps)}."
        )

    expected_ids = list(range(1, len(plan.steps) + 1))
    actual_ids = [step.id for step in plan.steps]
    if actual_ids != expected_ids:
        raise ValueError(
            f"Step IDs must be consecutive and one-based: {actual_ids!r}."
        )

    if not plan.goal.strip():
        raise ValueError("Plan goal must not be empty.")

    has_evidence = False
    draft_is_current = False
    awaiting_revision = False
    previous_agent = None
    editor_count = 0

    for step in plan.steps:
        agent = step.agent

        if not step.task.strip():
            raise ValueError(f"Step {step.id} task must not be empty.")

        if awaiting_revision and agent != "writer_agent":
            raise ValueError(
                f"Step {step.id}: editor feedback must be followed immediately "
                "by writer_agent."
            )

        if agent == "research_agent":
            has_evidence = True
            draft_is_current = False

        elif agent == "writer_agent":
            if not has_evidence:
                raise ValueError(
                    f"Step {step.id}: writer_agent requires prior research."
                )
            if previous_agent == "writer_agent":
                raise ValueError(
                    f"Step {step.id}: consecutive writer_agent steps should be merged."
                )
            draft_is_current = True
            awaiting_revision = False

        elif agent == "editor_agent":
            if not draft_is_current:
                raise ValueError(
                    f"Step {step.id}: editor_agent requires a current draft."
                )
            editor_count += 1
            awaiting_revision = True
            draft_is_current = False

        previous_agent = agent

    route = [step.agent for step in plan.steps]
    if editor_count != 1:
        raise ValueError("Plan must contain exactly one editor_agent step.")
    if route[-2:] != ["editor_agent", "writer_agent"]:
        raise ValueError("Plan must end with editor_agent -> writer_agent.")



In [30]:
def planner_agent(
    topic: str,
    model: str = MODEL_NAME,
    max_steps: int = 6,
    max_plan_attempts: int = 2,
    verbose: bool = True,
) -> TaskPlan:
    """
    Generate a structured execution plan for a research workflow.

    Args:
        topic: The research topic to investigate.
        model: The language model to use.
        max_steps: Maximum number of executable plan steps.
        max_plan_attempts: Maximum attempts after deterministic validation feedback.
        verbose: Whether to print concise execution logs.

    Returns:
        A validated task plan with explicit agent assignments.
    """

    topic = topic.strip()

    if not topic:
        raise ValueError("Topic must not be empty.")

    if max_steps < 4:
        raise ValueError("max_steps must be at least 4 for this workflow.")

    if max_plan_attempts < 1:
        raise ValueError("max_plan_attempts must be at least 1.")

    if verbose:
        print("[PLANNER_AGENT] Starting planning.")
        print(f"[PLANNER_AGENT] Topic: {topic}")

    validation_feedback = None
    for attempt in range(1, max_plan_attempts + 1):
        planner_input = (
            f"Create a research plan with 4 to {max_steps} steps "
            f"for the following topic:\n\n{topic}"
        )
        if validation_feedback:
            planner_input += (
                "\n\nThe previous plan was rejected by the workflow validator: "
                f"{validation_feedback}\nReturn a corrected complete plan."
            )

        if verbose:
            print(f"[PLANNER_AGENT] Model call #{attempt}")

        response = client.responses.parse(
            model=model,
            instructions=PLANNER_INSTRUCTIONS,
            input=planner_input,
            text_format=TaskPlan,
        )
        plan = response.output_parsed

        if plan is None:
            validation_feedback = "The response was not a valid TaskPlan."
        else:
            try:
                validate_task_plan(plan, max_steps=max_steps)
            except ValueError as exc:
                validation_feedback = str(exc)
            else:
                if verbose:
                    route = " -> ".join(step.agent for step in plan.steps)
                    print("[PLANNER_AGENT] Planning completed.")
                    print(f"[PLANNER_AGENT] Total steps: {len(plan.steps)}")
                    print(f"[PLANNER_AGENT] Agent route: {route}")
                return plan

        if verbose:
            print(
                f"[PLANNER_AGENT] Plan attempt {attempt} rejected: "
                f"{validation_feedback}"
            )

    raise RuntimeError(
        f"Planner failed after {max_plan_attempts} attempts: {validation_feedback}"
    )

In [31]:
topic = "What are the recent developments in AI design patterns for improving the reliability of LLM agents?"
plan = planner_agent(topic)
print(plan.model_dump_json(indent=2))

[PLANNER_AGENT] Starting planning.
[PLANNER_AGENT] Topic: What are the recent developments in AI design patterns for improving the reliability of LLM agents?
[PLANNER_AGENT] Model call #1
[PLANNER_AGENT] Planning completed.
[PLANNER_AGENT] Total steps: 5
[PLANNER_AGENT] Agent route: research_agent -> research_agent -> writer_agent -> editor_agent -> writer_agent
{
  "goal": "Produce a concise, evidence-backed plan outlining the latest AI design patterns for improving the reliability of LLM agents, identify gaps, and deliver a publication-ready report for researchers and practitioners.",
  "steps": [
    {
      "id": 1,
      "agent": "research_agent",
      "task": "Evidence area 1: Identify and collect the latest AI design patterns that aim to improve reliability of LLM agents. Gather sources from arXiv, conference papers, industry reports, standards, and blog posts; extract definitions, core claims, representative examples, and implementation notes; provide structured notes with cit

## 4.2 Research Agent

In [32]:
RESEARCH_INSTRUCTIONS = """
You are the Research Agent in a multi-agent deep research system.

Execute exactly one research step assigned by the Planner Agent.

Research requirements:

- Focus only on the supplied task step.
- Do not create or modify the task plan.
- Do not answer the user's overall question.
- Use the available tools to collect evidence.
- Use tavily_search_tool for recent developments, documentation, websites,
  articles, and general web information.
- Use arxiv_search_tool for papers, methods, experiments, and benchmarks.
- Use plain semantic keywords for arXiv. Do not use URLs, site operators,
  the word arXiv, or web-style date ranges such as 2020..2026.
- Refine a search query only when its results leave a material evidence gap.
- Prefer 3 to 5 focused searches over many overlapping searches.
- Do not repeat searches that differ only slightly.
- Base factual claims on tool results rather than unsupported memory.
- Do not invent sources, titles, authors, dates, or URLs.
- Preserve source URLs exactly as returned by the tools.
- Prefer primary and academic sources when available.
- Clearly identify incomplete or conflicting evidence.
- Stop once the supplied task can be answered with sufficient relevant evidence.
- After collecting several useful primary and web sources, synthesize the
  findings instead of continuing to search.

Return concise research notes in Markdown using this structure:

## Summary

A concise summary of the research result.

## Findings

Evidence-backed findings with source links.

## Sources

A list of source titles and URLs.

## Limitations

Missing, uncertain, or conflicting information.

Return only the structured research notes. Do not include conversational
text or a user-facing final answer.

Your output will be consumed by later agents through the execution history.
""".strip()

In [33]:
def execute_tool_call(tool_call: Any) -> dict[str, Any]:
    """Execute one model tool call and return a Responses API observation."""

    try:
        result = execute_tool(
            name=tool_call.name,
            arguments=json.loads(tool_call.arguments),
        )
        tool_errors = [
            str(item["error"])
            for item in result
            if isinstance(item, dict) and "error" in item
        ] if isinstance(result, list) else []
        if tool_errors:
            raise RuntimeError("; ".join(tool_errors))
        payload = {"ok": True, "result": result}
    except Exception as exc:  # noqa: BLE001 - return failures to the model
        payload = {"ok": False, "error": f"{type(exc).__name__}: {exc}"}

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(payload, ensure_ascii=False),
    }


def run_research_tool_loop(
    user_input: str,
    model: str,
    max_tool_rounds: int,
    verbose: bool,
) -> str:
    """Run the model-tool-observation loop and force bounded synthesis."""

    if max_tool_rounds < 0:
        raise ValueError("max_tool_rounds must not be negative.")

    input_items: list[Any] = [{"role": "user", "content": user_input}]

    for tool_round in range(max_tool_rounds + 1):
        budget_exhausted = tool_round == max_tool_rounds
        request_input = input_items

        if budget_exhausted:
            request_input = [
                *input_items,
                {
                    "role": "user",
                    "content": (
                        "The tool budget is exhausted. Do not request more tools. "
                        "Produce the final research notes from collected evidence."
                    ),
                },
            ]

        if verbose:
            print(f"[RESEARCH_AGENT] Model call #{tool_round + 1}")

        response = client.responses.create(
            model=model,
            instructions=RESEARCH_INSTRUCTIONS,
            input=request_input,
            tools=TOOLS,
            tool_choice="none" if budget_exhausted else "auto",
            parallel_tool_calls=False,
        )
        input_items.extend(response.output)
        tool_calls = [
            item for item in response.output if item.type == "function_call"
        ]

        if not tool_calls:
            result = response.output_text.strip()
            if not result:
                raise RuntimeError("Research agent returned an empty result.")
            return result

        if budget_exhausted:
            raise RuntimeError("Research agent requested tools after its budget expired.")

        if verbose:
            names = ", ".join(tool_call.name for tool_call in tool_calls)
            print(f"[RESEARCH_AGENT] Tool round #{tool_round + 1}: {names}")

        input_items.extend(execute_tool_call(tool_call) for tool_call in tool_calls)

    raise RuntimeError("Research loop ended without a final result.")


def research_agent(
    user_request: str,
    step: PlanStep,
    history: list[dict[str, Any]],
    model: str = MODEL_NAME,
    max_tool_rounds: int = 8,
    verbose: bool = True,
) -> str:
    """Adapt one research plan step to the reusable tool loop."""

    if step.agent != "research_agent":
        raise ValueError(
            f"Task step {step.id!r} is assigned to {step.agent!r}, "
            "not 'research_agent'."
        )

    if verbose:
        print(f"[RESEARCH_AGENT] Starting step {step.id}: {step.task}")

    result = run_research_tool_loop(
        user_input=json.dumps(
            {
                "user_request": user_request,
                "task": step.task,
                "execution_history": history,
            },
            ensure_ascii=False,
        ),
        model=model,
        max_tool_rounds=max_tool_rounds,
        verbose=verbose,
    )

    if verbose:
        print(f"[RESEARCH_AGENT] Step {step.id} completed: {len(result)} characters")

    return result

In [34]:
research_step = next(step for step in plan.steps if step.agent == "research_agent")

research_result = research_agent(
    user_request=topic,
    step=research_step,
    history=[],
)

[RESEARCH_AGENT] Starting step 1: Evidence area 1: Identify and collect the latest AI design patterns that aim to improve reliability of LLM agents. Gather sources from arXiv, conference papers, industry reports, standards, and blog posts; extract definitions, core claims, representative examples, and implementation notes; provide structured notes with citations, source metadata (title, authors, venue, year, URL), and a brief assessment of evidence quality.
[RESEARCH_AGENT] Model call #1
[RESEARCH_AGENT] Tool round #1: arxiv_search_tool
[RESEARCH_AGENT] Model call #2
[RESEARCH_AGENT] Tool round #2: tavily_search_tool
[RESEARCH_AGENT] Model call #3
[RESEARCH_AGENT] Step 1 completed: 9914 characters


NameError: name 'show' is not defined

In [35]:
Markdown(research_result)

## Summary

Collected recent evidence on AI design patterns aimed at improving the reliability of LLM agents. Evidence spans arXiv papers (architectural/runtime patterns, agentic governance, latency–reliability aids, and domain-specific collaborations) and practitioner blog posts/industry patterns. Key themes include formalized runtime contracts, multi-agent coordination, plan reuse to reduce latency, role/guild rotation and information fusion, and human-in-the-loop or moderator-based reliability enhancements.

## Findings

- Stochastic-Deterministic Boundary (SDB) as a reliability primitive for production LLM agents
  - Evidence: A Methodology for Selecting and Composing Runtime Architecture Patterns for Production LLM Agents (arXiv, 2026)
  - Core claims:
    - Introduces the stochastic-deterministic boundary (SDB) as the contract that turns an LLM output into a system action.
    - Proposes a four-part contract among proposer, verifier, commit, and reject signal.
    - Organizes runtime concerns into Coordination, State, and Control, around SDB.
    - Presents six runtime patterns to compose SDB across conversational, autonomous, and long-horizon agents: hierarchical delegation, scatter-gather plus saga, event-driven sequencing, shared state machine, supervisor plus gate, and human in the loop.
    - Five-step methodology for pattern selection; diagnostic for failures; identifies a failure mode called replay divergence.
    - A stylized reliability decomposition separating per-call model variance from architectural momentum.
    - Includes a runnable reference implementation for a 90-day contract-renewal agent.
  - Representative examples/notes:
    - Emphasizes planning the boundary between stochastic model outputs and deterministic workflows.
    - Concrete patterns provide reusable architectural templates for production agents.
  - Evidence quality: arXiv preprint (peer review status unknown). Good for architectural guidance and milestones; needs replication and broader evaluation.
  - URL: http://arxiv.org/abs/2605.20173v2

- Design-patterns for production/enterprise LLM agent architectures (tiered, governance-aware patterns)
  - Evidence: Architecting Agentic Communities using Design Patterns (arXiv, 2026)
  - Core claims:
    - Proposes three tiers of patterns: LLM Agents (task-specific automation), Agentic AI (adaptive goal-seekers), and Agentic Communities (governed ecosystems with formal roles/protocols).
    - Focuses on agentic communities with coordination frameworks, governance, and formal collaboration agreements between AI agents and humans.
    - Grounded in distributed-systems coordination principles with formal verification capabilities.
    - Validates via a clinical trial matching case study.
  - Representative examples/notes:
    - Emphasizes governance, accountability, and verifiable inter-agent communication and intent modeling.
  - Evidence quality: arXiv preprint; domain-focused case study; preliminary architectural guidance.
  - URL: http://arxiv.org/abs/2601.03624v3

- Plan reuse to improve responsiveness/reliability in LLM-driven agents
  - Evidence: A Plan Reuse Mechanism for LLM-Driven Agent (arXiv, 2025)
  - Core claims:
    - Proposes AgentReuse to reuse previously generated plans for similar user requests to cut latency.
    - Uses intent classification to assess similarity between requests and enable plan reuse.
  - Representative results:
    - 93% effective plan reuse rate; F1 = 0.9718; accuracy = 0.9459 in similarity evaluation.
    - Latency reduced by 93.12% vs baselines without reuse.
  - Evidence quality: arXiv preprint; compelling latency results on real-world data; needs broader validation across domains.
  - URL: http://arxiv.org/abs/2512.21309v2

- Multi-agent collaboration in medicine to improve reliability and reduce hallucinations
  - Evidence: MedAide: Information Fusion and Anatomy of Medical Intents via LLM-based Agent Collaboration (arXiv, 2024)
  - Core claims:
    - Proposes multi-agent collaboration with intent-aware information fusion and structured decomposition of complex medical intents.
    - Introduces a regularization-guided module combining syntactic constraints with retrieval-augmented generation (RAG) for decomposition.
    - Dynamic intent prototype matching for adaptive intent recognition/updating in multi-round dialogues.
    - Rotation agent collaboration to dynamically rotate roles and fuse information across specialized medical agents.
  - Representative results:
    - Outperforms current LLMs on four medical benchmarks; improves medical proficiency and strategic reasoning (expert evaluation included).
  - Evidence quality: arXiv preprint; domain-specific results with automated metrics and expert assessment; generalizability beyond medical domain requires more work.
  - URL: http://arxiv.org/abs/2410.12532v3

- Reliability-focused focus-group simulation and moderation using LLMs
  - Evidence: Focus Agent: LLM-Powered Virtual Focus Group (arXiv, 2024)
  - Core claims:
    - Introduces an LLM-powered Focus Agent to simulate focus groups and act as moderator.
    - Assesses data quality across five human focus groups (23 participants) and AI-simulated sessions.
    - Quantitative analyses indicate Focus Agent-generated data can resemble human-consensus opinions; identifies improvements when LLMs act as moderators.
  - Evidence quality: arXiv preprint; exploratory study, preliminary findings; broader validation needed.
  - URL: http://arxiv.org/abs/2409.01907v1

- Practitioner design patterns for reliable AI agents (industry-oriented, non-peer-reviewed)
  - Evidence: 7 Practical Design Patterns for Agentic Systems (MongoDB blog)
  - Core claims:
    - Presents patterns to keep systems reliable despite LLM nondeterminism.
    - Highlights human-in-the-loop at critical points, decomposition into steps, and deterministic validation gates.
  - Notes: Practitioner-oriented, not peer-reviewed; useful for implementation guidance.
  - URL: https://www.mongodb.com/resources/basics/artificial-intelligence/agentic-systems

- Practitioner design patterns for reliability in agentic AI (SAP)
  - Evidence: Tame Your Agents: 10 Design Patterns for Reliable Agentic AI (SAP Community)
  - Core claims:
    - Covers patterns including fixed sequencing (draft → critique → revise), agentic RAG, and validation gates.
  - Notes: Industry blog; synthesis of multiple prior works; practical guidance.
  - URL: https://community.sap.com/t5/technology-blog-posts-by-sap/tame-your-agents-10-design-patterns-for-reliable-agentic-ai/ba-p/14424874

- General guidance on building reliable AI agent architectures (industry blog)
  - Evidence: AI Agent Design: How to Build Reliable AI Agent Architecture (Comet)
  - Core claims:
    - Emphasizes modular design, observability, and feedback loops for improving reliability.
    - Discusses LLM as judge for evaluation and observability practices.
  - Notes: Industry blog; practical perspective.
  - URL: https://www.comet.com/site/blog/ai-agent-design

- Additional design-pattern context (Databricks)
  - Evidence: Agent System Design Patterns (Databricks)
  - Core claims:
    - Presents patterns to mitigate common failure modes (tool miscalls, drift, cost spikes) and to build reliable agent systems.
  - Notes: Documentation-focused, enterprise-oriented guidance.
  - URL: https://docs.databricks.com/aws/en/agents/agent-system-design-patterns

- General reliability design patterns (DEV)
  - Evidence: 4 Design Patterns That Make AI Agents Actually Reliable (DEV)
  - Core claims:
    - Emphasizes decomposition into steps, constraints, and concrete validation to improve reliability.
  - Notes: Community blog post; practical takeaways.
  - URL: https://dev.to/wassimchegham/4-design-patterns-that-make-ai-agents-actually-reliable-lgc

## Sources

ArXiv / Papers
- A Methodology for Selecting and Composing Runtime Architecture Patterns for Production LLM Agents — http://arxiv.org/abs/2605.20173v2
- Focus Agent: LLM-Powered Virtual Focus Group — http://arxiv.org/abs/2409.01907v1
- Architecting Agentic Communities using Design Patterns — http://arxiv.org/abs/2601.03624v3
- A Plan Reuse Mechanism for LLM-Driven Agent — http://arxiv.org/abs/2512.21309v2
- MedAide: Information Fusion and Anatomy of Medical Intents via LLM-based Agent Collaboration — http://arxiv.org/abs/2410.12532v3

Industry/Blog (Web)
- 7 Practical Design Patterns for Agentic Systems — https://www.mongodb.com/resources/basics/artificial-intelligence/agentic-systems
- Tame Your Agents: 10 Design Patterns for Reliable Agentic AI — https://community.sap.com/t5/technology-blog-posts-by-sap/tame-your-agents-10-design-patterns-for-reliable-agentic-ai/ba-p/14424874
- AI Agent Design: How to Build Reliable AI Agent Architecture — https://www.comet.com/site/blog/ai-agent-design
- Agent System Design Patterns — https://docs.databricks.com/aws/en/agents/agent-system-design-patterns
- 4 Design Patterns That Make AI Agents Actually Reliable — https://dev.to/wassimchegham/4-design-patterns-that-make-ai-agents-actually-reliable-lgc

## Limitations

- Most arXiv sources are preprints with limited or no peer review; results are early-stage and may require replication and broader cross-domain validation.
- Blog/industry posts provide practical guidance but are not peer-reviewed; patterns may be context-specific and not universally validated.
- Some sources emphasize architecture and governance patterns more than formal reliability metrics; broader consensus and standards are still evolving.
- Domain generalizability varies: medical-domain results (MedAide) are compelling but domain-specific; reliability in other domains remains to be demonstrated.
- The set of patterns shown may evolve as new methods (e.g., advanced verification, external tools, safety layers) mature; ongoing monitoring of the literature is needed.

## 4.3 Writer Agent

In [37]:
WRITER_INSTRUCTIONS = """
You are the Writer Agent in a multi-agent deep research system.

Execute exactly one writing step assigned by the Planner Agent.

The supplied execution history may contain:

- Research notes from one or more research steps
- An earlier draft
- Editorial feedback

Writing requirements:

- Follow the supplied task exactly.
- Use only the information contained in the execution history.
- Do not invent facts, sources, titles, dates, or URLs.
- Preserve source URLs exactly as provided.
- Clearly distinguish established findings from uncertain conclusions.
- Synthesize overlapping research rather than repeating it.
- Resolve minor inconsistencies when the supplied evidence allows it.
- Preserve important limitations and disagreements between sources.
- Use descriptive headings and a logical narrative structure.
- Include citations close to the claims they support.
- Avoid discussing the internal agent workflow in the report.
- Do not describe an arXiv preprint as peer reviewed unless the evidence says so.

If the task requests an initial draft:

- Synthesize the supplied research results into a coherent report.
- Ensure the report addresses the original user request.

If the task requests a revision:

- Use both the earlier draft and the editorial feedback.
- Address all required editorial changes.
- Return the complete revised report, not a list of modifications.
- When report_requirements are provided, follow their word limits.
- Use one top-level Markdown title and second-level section headings.
- Include ## Abstract, ## Executive Summary, and ## References sections.
- Deduplicate references and preserve available source metadata.
- Do not include a change summary or editorial commentary.

Return only the requested report content in Markdown.
Do not include a preface, afterword, change summary, follow-up question,
offer to continue, or any conversational text addressed to the user.
When producing the final report, end the document with the References section.
""".strip()

In [38]:
def writer_agent(
    user_request: str,
    step: PlanStep,
    history: list[dict[str, Any]],
    model: str = MODEL_NAME,
    report_requirements: dict[str, int] | None = None,
    verbose: bool = True,
) -> str:
    if step.agent != "writer_agent":
        raise ValueError(
            f"Task step {step.id!r} is assigned to "
            f"{step.agent!r}, not 'writer_agent'."
        )

    if verbose:
        print(f"[{step.agent.upper()}] Starting step {step.id}")
        print(f"[{step.agent.upper()}] Task: {step.task}")
        print(f"[{step.agent.upper()}] Context steps: {len(history)}")
        print(f"[{step.agent.upper()}] Model call #1")

    writer_request = {
        "user_request": user_request,
        "task_step": step.model_dump(mode="json"),
        "execution_history": history,
        "report_requirements": report_requirements,
    }

    response = client.responses.create(
        model=model,
        instructions=WRITER_INSTRUCTIONS,
        input=json.dumps(
            writer_request,
            ensure_ascii=False,
        ),
    )

    result = response.output_text.strip()

    if not result:
        raise RuntimeError(
            f"Writer agent returned no result for step {step.id!r}."
        )

    if verbose:
        print(f"[{step.agent.upper()}] Writing step {step.id} completed.")
        print(f"[{step.agent.upper()}] Output characters: {len(result)}")

    return result

In [39]:
writer_step = next(
    step
    for step in plan.steps
    if step.agent == "writer_agent"
)

writer_history = [
    {
        "step_id": research_step.id,
        "agent": research_step.agent,
        "task": research_step.task,
        "result": research_result,
    }
]

writer_result = writer_agent(
    user_request=topic,
    step=writer_step,
    history=writer_history,
)

[WRITER_AGENT] Starting step 3
[WRITER_AGENT] Task: Draft initial synthesis: produce a cohesive narrative combining findings from Steps 1-2 into a taxonomy of AI design patterns for reliability in LLM agents. For each pattern, summarize definition, typical use cases, evidence strength, representative examples, and caveats; outline open gaps and tensions; ensure clear citation mapping.
[WRITER_AGENT] Context steps: 1
[WRITER_AGENT] Model call #1
[WRITER_AGENT] Writing step 3 completed.
[WRITER_AGENT] Output characters: 15757


In [40]:
Markdown(writer_result)

# Recent Developments in AI Design Patterns for Reliability in LLM Agents: A Synthesis

This synthesis collates findings from recent design-pattern literature and preprints to present a cohesive taxonomy of patterns aimed at improving the reliability of LLM agents. The evidence base spans arXiv preprints, industry blogs, and enterprise guidance. Across sources, common threads include explicit runtime contracts, governance-aware architectures, reuse and planning efficiency, multi-agent collaboration, moderator-like reliability enhancements, and practitioner-oriented patterns focused on observability and validation gates. Note that most arXiv sources are preprints with limited or no peer review, and industry/blog sources offer practical guidance that may not be universally validated.

## Pattern Taxonomy and Synthesis

### 1) Stochastic-Deterministic Boundary (SDB) and Composable Runtime Patterns
Definition
- A contract that turns an LLM output into a system action, organized around a four-part contract among proposer, verifier, commit, and reject signal. Runtime concerns are organized into Coordination, State, and Control around this boundary. The design envisions six reusable runtime patterns to compose SDB across conversational, autonomous, and long-horizon agents: hierarchical delegation, scatter-gather plus saga, event-driven sequencing, shared state machine, supervisor plus gate, and human in the loop. A five-step pattern-selection methodology and a diagnostic for failures (including a replay divergence) are proposed, along with a stylized reliability decomposition that separates per-call model variance from architectural momentum, plus a runnable reference implementation for a 90-day contract-renewal agent. [A Methodology for Selecting and Composing Runtime Architecture Patterns for Production LLM Agents, arXiv:2605.20173v2]

Evidence strength and caveats
- Evidence quality: arXiv preprint; provides architectural guidance and milestones but lacks broad replication and peer-reviewed validation. [2605.20173v2]
- Caveats: Early-stage design guidance; real-world generalizability and cross-domain validation remain to be demonstrated.

Representative notes
- Concrete patterns serve as reusable templates for production agents; the boundary helps separate stochastic model outputs from deterministic workflows. [2605.20173v2]

Open gaps and tensions
- Need for cross-domain replication, tooling integration, and standardized safety/regulatory alignment across industries.

Citations
- (A Methodology for Selecting and Composing Runtime Architecture Patterns for Production LLM Agents, arXiv:2605.20173v2)

---

### 2) Tiered, Governance-Aware Agent Architectures (Agentic Communities)
Definition
- A design pattern framework organizing patterns across three tiers: LLM Agents (task-specific automation), Agentic AI (adaptive goal-seekers), and Agentic Communities (governed ecosystems with formal roles and inter-agent human coordination). Emphasizes governance, accountability, verifiable inter-agent communication, and intent modeling, grounded in distributed-systems coordination principles and formal verification concepts. A clinical trial–adjacent case study is used to illustrate governance-style validation. [Architecting Agentic Communities using Design Patterns, arXiv:2601.03624v3]

Evidence strength and caveats
- Evidence quality: arXiv preprint; domain-focused case study; preliminary architectural guidance. [2601.03624v3]
- Caveats: Early-stage; broader cross-domain validation and standards development are needed.

Representative notes
- Governance, verifiability, and formal collaboration agreements are foregrounded to increase reliability and responsibility in multi-agent ecosystems. [2601.03624v3]

Open gaps and tensions
- Interoperability standards across agents and humans; scalable verification mechanisms; alignment across Domains and regulatory regimes.

Citations
- (Architecting Agentic Communities using Design Patterns, arXiv:2601.03624v3)

---

### 3) Plan Reuse to Improve Responsiveness and Reliability (AgentReuse)
Definition
- A plan reuse mechanism that reuses previously generated plans for similar user requests to reduce latency. The approach relies on intent classification to determine similarity and enable reuse. Reported benefits include substantial latency reductions and high plan-reuse effectiveness in empirical evaluations. [A Plan Reuse Mechanism for LLM-Driven Agent, arXiv:2512.21309v2]

Evidence strength and caveats
- Evidence quality: arXiv preprint; compelling latency results on real-world data; broader domain validation is still needed. [2512.21309v2]
- Caveats: Reuse may risk context drift, outdated plans, or incorrect applicability if similarity judgments are imperfect.

Representative notes
- Emphasizes responsiveness gains by leveraging prior work, aligning with reliability goals when latency is critical. [2512.21309v2]

Open gaps and tensions
- Robust similarity metrics, safety checks for reused plans, and governance of plan freshness and applicability.

Citations
- (A Plan Reuse Mechanism for LLM-Driven Agent, arXiv:2512.21309v2)

---

### 4) MedAide: Multi-Agent Collaboration with Intent Fusion in Medicine
Definition
- A multi-agent collaboration approach with intent-aware information fusion and structured decomposition of complex medical intents. Introduces rotation-based agent collaboration to dynamically rotate roles and fuse information across specialized medical agents, combined with regularization-guided decomposition that blends syntactic constraints with retrieval-augmented generation (RAG). Dynamic intent prototype matching supports adaptive intent recognition/updating across multi-round dialogues. Evidence reports improvement over current LLM baselines on four medical benchmarks with expert evaluation. [MedAide: Information Fusion and Anatomy of Medical Intents via LLM-based Agent Collaboration, arXiv:2410.12532v3]

Evidence strength and caveats
- Evidence quality: arXiv preprint; domain-specific results with automated metrics and expert assessment; generalizability beyond medicine requires more work. [2410.12532v3]
- Caveats: Medical domain focus; results may not generalize to non-medical domains without adaptation.

Representative notes
- Intent-aware information fusion and role rotation enable more reliable medical reasoning in multi-agent settings. [2410.12532v3]

Open gaps and tensions
- Cross-domain applicability, safety/regulatory considerations, and validation in diverse clinical contexts.

Citations
- (MedAide: Information Fusion and Anatomy of Medical Intents via LLM-based Agent Collaboration, arXiv:2410.12532v3)

---

### 5) Focus Agent: Moderator-Driven Focus Group Simulations for Reliability
Definition
- An LLM-powered Focus Agent designed to simulate focus groups and act as a moderator. It assesses data quality across human focus groups and AI-simulated sessions; results suggest Focus Agent–moderated data approaches yield data that resemble human-consensus opinions and yield reliability gains. [Focus Agent: LLM-Powered Virtual Focus Group, arXiv:2409.01907v1]

Evidence strength and caveats
- Evidence quality: arXiv preprint; exploratory study with a limited sample; broader validation needed. [2409.01907v1]
- Caveats: Preliminary findings; generalizability to other moderation tasks or domains is unproven.

Representative notes
- Moderation and structured data quality assessments are proposed as reliability-enhancing roles for LLMs in social/compositional tasks. [2409.01907v1]

Open gaps and tensions
- Need for larger-scale studies, domain diversification, and rigorous evaluation against human moderators.

Citations
- (Focus Agent: LLM-Powered Virtual Focus Group, arXiv:2409.01907v1)

---

### 6) Practitioner Design Patterns for Reliable Agentic Systems (Industry-Facing)
Definition
- Industry-oriented design patterns that help keep AI agents reliable despite LLM nondeterminism, emphasizing human-in-the-loop at critical points, step decompositions, and deterministic validation gates. Sources include practitioner-focused patterns from MongoDB, SAP, Comet, Databricks, and DEV. Representative concepts include explicit gating, modular decomposition, observability, and judgmental evaluation by LLMs. [Industry sources: MongoDB, SAP, Comet, Databricks, DEV]

Evidence strength and caveats
- Evidence quality: Industry blogs and documentation; not peer-reviewed; patterns are practice-oriented and context-dependent. [MongoDB blog; SAP blog; Comet; Databricks; DEV]
- Caveats: Context-specific guidance; may not generalize across all domains or compliance regimes.

Representative notes
- Emphasize human-in-the-loop at critical junctures, decomposition into orderly steps, deterministic validation gates, and observability to improve reliability in real-world deployments. Examples span several providers and documentation resources. [MongoDB; SAP; Comet; Databricks; DEV]

Open gaps and tensions
- Need for consolidated standards and cross-domain validation; alignment with regulatory and safety requirements; variance in implementation details across vendors.

Citations
- (7 Practical Design Patterns for Agentic Systems, MongoDB)
- (Tame Your Agents: 10 Design Patterns for Reliable Agentic AI, SAP)
- (AI Agent Design: How to Build Reliable AI Agent Architecture, Comet)
- (Agent System Design Patterns, Databricks)
- (4 Design Patterns That Make AI Agents Actually Reliable, DEV)

---

### 7) General Reliability Guidance and Observability in Agent Architectures (Industry Guidance)
Definition
- General guidance on building reliable AI agent architectures emphasizing modular design, observability, and feedback loops; positions LLMs as evaluators/judges for certain reliability tasks and stresses the importance of observability practices. [AI Agent Design: How to Build Reliable AI Agent Architecture, Comet]

Evidence strength and caveats
- Evidence quality: Industry blog guidance; practical perspective rather than formal validation. [Comet]
- Caveats: Context-specific and not peer-reviewed; needs integration with formal reliability metrics.

Citations
- (AI Agent Design: How to Build Reliable AI Agent Architecture, Comet)

---

### 8) Additional Context: Enterprise-Driven Pattern Catalogs (Databricks)
Definition
- Enterprise-oriented design patterns that address common failure modes in agent systems (tool miscalls, model drift, cost spikes) and provide patterns for building more reliable agent systems. [Agent System Design Patterns, Databricks]

Evidence strength and caveats
- Evidence quality: Documentation-focused guidance; enterprise-oriented and descriptive, not peer-reviewed. [Databricks]
- Caveats: Pattern catalogs may reflect vendor-specific viewpoints; cross-domain applicability should be evaluated.

Citations
- (Agent System Design Patterns, Databricks)

---

### 9) General Reliability Patterns in the Community (DEV)
Definition
- Community-driven patterns emphasizing decomposition into steps, constraints, and concrete validation to improve reliability in AI agents. [4 Design Patterns That Make AI Agents Actually Reliable, DEV]

Evidence strength and caveats
- Evidence quality: Community blog post; practical guidance rather than formal validation. [DEV]
- Caveats: Varying quality and applicability; not a formal evidence base.

Citations
- (4 Design Patterns That Make AI Agents Actually Reliable, DEV)

---

## Cross-Cutting Gaps, Tensions, and Research Needs

- Evidence quality and generalizability
  - A large portion of the foundational results come from arXiv preprints and industry blogs. While these sources provide valuable architectural guidance and practical patterns, many lack broad peer-reviewed validation or cross-domain replication. This necessitates cautious interpretation and targeted cross-domain evaluation. [2605.20173v2; 2601.03624v3; 2512.21309v2; 2410.12532v3; 2409.01907v1]

- Domain-generalizability vs. domain-specific results
  - Medical-domain results (MedAide) are compelling but domain-specific; applicability to other domains remains an open question. Governance and community-pattern approaches require testing across regulatory contexts and different stakeholder ecosystems. [2410.12532v3]

- Governance, standards, and interoperability
  - Pattern families emphasize governance and inter-agent collaboration, but standardized interoperability across agents, humans, and external tools remains an area for development. [2601.03624v3; MongoDB; SAP; Comet; Databricks]

- Safety, ethics, and regulatory alignment
  - Several patterns imply human-in-the-loop and validation gates, yet systematic integration with safety frameworks and regulatory compliance is not uniformly addressed across sources.

- Tooling maturity and integration
  - Many concepts rely on orchestration, state machines, and plan repositories; practical adoption depends on tooling ecosystems and integration with existing platforms, which vary by source.

## Reference Map (Selected Sources)

- A Methodology for Selecting and Composing Runtime Architecture Patterns for Production LLM Agents. arXiv:2605.20173v2. http://arxiv.org/abs/2605.20173v2
- Architecting Agentic Communities using Design Patterns. arXiv:2601.03624v3. http://arxiv.org/abs/2601.03624v3
- A Plan Reuse Mechanism for LLM-Driven Agent. arXiv:2512.21309v2. http://arxiv.org/abs/2512.21309v2
- MedAide: Information Fusion and Anatomy of Medical Intents via LLM-based Agent Collaboration. arXiv:2410.12532v3. http://arxiv.org/abs/2410.12532v3
- Focus Agent: LLM-Powered Virtual Focus Group. arXiv:2409.01907v1. http://arxiv.org/abs/2409.01907v1

Industry/Blog/Documentation (URLs preserved exactly)
- 7 Practical Design Patterns for Agentic Systems. https://www.mongodb.com/resources/basics/artificial-intelligence/agentic-systems
- Tame Your Agents: 10 Design Patterns for Reliable Agentic AI. https://community.sap.com/t5/technology-blog-posts-by-sap/tame-your-agents-10-design-patterns-for-reliable-agentic-ai/ba-p/14424874
- AI Agent Design: How to Build Reliable AI Agent Architecture. https://www.comet.com/site/blog/ai-agent-design
- Agent System Design Patterns. https://docs.databricks.com/aws/en/agents/agent-system-design-patterns
- 4 Design Patterns That Make AI Agents Actually Reliable. https://dev.to/wassimchegham/4-design-patterns-that-make-ai-agents-actually-reliable-lgc

## References

- A Methodology for Selecting and Composing Runtime Architecture Patterns for Production LLM Agents — arXiv, 2026. http://arxiv.org/abs/2605.20173v2
- Architecting Agentic Communities using Design Patterns — arXiv, 2026. http://arxiv.org/abs/2601.03624v3
- A Plan Reuse Mechanism for LLM-Driven Agent — arXiv, 2025. http://arxiv.org/abs/2512.21309v2
- MedAide: Information Fusion and Anatomy of Medical Intents via LLM-based Agent Collaboration — arXiv, 2024. http://arxiv.org/abs/2410.12532v3
- Focus Agent: LLM-Powered Virtual Focus Group — arXiv, 2024. http://arxiv.org/abs/2409.01907v1

- 7 Practical Design Patterns for Agentic Systems — MongoDB Blog. https://www.mongodb.com/resources/basics/artificial-intelligence/agentic-systems
- Tame Your Agents: 10 Design Patterns for Reliable Agentic AI — SAP Community. https://community.sap.com/t5/technology-blog-posts-by-sap/tame-your-agents-10-design-patterns-for-reliable-agentic-ai/ba-p/14424874
- AI Agent Design: How to Build Reliable AI Agent Architecture — Comet. https://www.comet.com/site/blog/ai-agent-design
- Agent System Design Patterns — Databricks. https://docs.databricks.com/aws/en/agents/agent-system-design-patterns
- 4 Design Patterns That Make AI Agents Actually Reliable — DEV. https://dev.to/wassimchegham/4-design-patterns-that-make-ai-agents-actually-reliable-lgc

## 4.4 Editor Agent

In [41]:
EDITOR_INSTRUCTIONS = """
You are the Editor Agent in a multi-agent deep research system.

Critically review exactly one draft according to the step assigned by the
Planner Agent.

Review requirements:

- Evaluate whether the draft addresses the original user request.
- Evaluate factual support using only the supplied content.
- Identify unsupported or overstated claims.
- Check whether citations are placed near the claims they support.
- Identify missing citations, suspicious citations, and broken logical links.
- Check coverage, structure, clarity, concision, and internal consistency.
- Identify important research findings that were omitted or distorted.
- Distinguish required corrections from optional improvements.
- Do not invent facts or sources.
- Do not rewrite the entire report.
- Make every criticism specific and actionable.
- Keep the review concise and prioritize no more than ten required changes.

Return the review in Markdown using this structure:

## Verdict

Use either APPROVE or REVISE and provide a short explanation.

## Strengths

List the strongest aspects of the draft.

## Required Changes

List factual, structural, or coverage problems that must be fixed.

## Citation Issues

List unsupported claims and citation problems.

## Optional Improvements

List non-essential improvements.

Return only the structured editorial review. Do not include a rewritten
report or conversational text.

Your review will be consumed by the Writer Agent during revision.
""".strip()

In [42]:
def editor_agent(
    user_request: str,
    step: PlanStep,
    history: list[dict[str, Any]],
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> str:
    if step.agent != "editor_agent":
        raise ValueError(
            f"Task step {step.id!r} is assigned to "
            f"{step.agent!r}, not 'editor_agent'."
        )

    if verbose:
        print(f"[{step.agent.upper()}] Starting step {step.id}")
        print(f"[{step.agent.upper()}] Task: {step.task}")
        print(f"[{step.agent.upper()}] Context steps: {len(history)}")
        print(f"[{step.agent.upper()}] Model call #1")

    editor_request = {
        "user_request": user_request,
        "task_step": step.model_dump(mode="json"),
        "execution_history": history,
    }

    response = client.responses.create(
        model=model,
        instructions=EDITOR_INSTRUCTIONS,
        input=json.dumps(
            editor_request,
            ensure_ascii=False,
        ),
    )

    result = response.output_text.strip()

    if not result:
        raise RuntimeError(
            f"Editor agent returned no result for step {step.id!r}."
        )

    if verbose:
        print(f"[{step.agent.upper()}] Editing step {step.id} completed.")
        print(f"[{step.agent.upper()}] Output characters: {len(result)}")

    return result

In [ ]:
editor_step = next(
    step
    for step in plan.steps
    if step.agent == "editor_agent"
)

editor_history = writer_history + [
    {
        "step_id": writer_step.id,
        "agent": writer_step.agent,
        "task": writer_step.task,
        "result": writer_result,
    }
]

editor_result = editor_agent(
    user_request=topic,
    step=editor_step,
    history=editor_history,
)

[EDITOR_AGENT] Starting step 4
[EDITOR_AGENT] Task: Review the current draft for coverage and structure: verify that all design-pattern categories are represented, check evidence support and citations, assess clarity and flow, identify missing sources or gaps, and ensure alignment with the original request.
[EDITOR_AGENT] Context steps: 2
[EDITOR_AGENT] Model call #1
[EDITOR_AGENT] Editing step 4 completed.
[EDITOR_AGENT] Output characters: 4664


NameError: name 'show' is not defined

In [44]:
Markdown(editor_result)

## Verdict

APPROVE

Reason: The draft delivers a coherent taxonomy of recent AI design patterns aimed at improving the reliability of LLM agents and ties each pattern to specific sources. It addresses the user’s request, assesses evidence strength, and highlights gaps and caveats. Some language and citation placement could be tightened, but the overall structure, coverage, and critical framing are solid.

## Strengths

- Clear taxonomy of patterns relevant to reliability (runtime contracts, governance, plan reuse, multi-agent collaboration, moderator-like reliability, observability, and industry guidance).
- Pattern definitions paired with typical use cases, representative notes, evidence strength, and caveats.
- Explicit cross-pattern gaps, tensions, and research needs, including domain-generalizability and safety/regulatory considerations.
- Structured mapping of sources (arXiv preprints, industry blogs, and documentation) with a transparent evidence ladder.
- Attention to evidence quality (peer-review status, domain limitations) and to avoiding overclaiming results.

## Required Changes

1) Calibrate claims to evidence strength
- Several results (e.g., plan reuse effectiveness, medical benchmark gains) are compelling but should be consistently framed as domain-specific or preprint-backed. Where claims are strong, add explicit caveats and attribute them to the source’s stated limitations.

2) Clarify and harmonize a potentially overstated pattern claim
- Pattern 2 mentions a “clinical trial adjacent case study.” If the source does not equate to a clinical trial, adjust language to “case study illustrating governance/verification in a healthcare context” to avoid overstatement.

3) Strengthen citation placement for claims
- Move inline citations so that each key claim (definition, representative results, and representative notes) is immediately followed by the corresponding source reference. In places, citations are present but not directly attached to the specific factual claim.

4) Consistency on evidence labeling across patterns
- Maintain uniform terminology: for all patterns, consistently label evidence strength (e.g., “evidence quality: arXiv preprint; limited replication”) in the same format and place.

5) Expand cross-domain safety and regulatory framing
- While governance and human-in-the-loop themes are mentioned, add a concise, explicit note on safety frameworks, external verification, and regulatory alignment as a cross-cutting requirement across patterns.

6) Omit or explicitly flag domain-limited claims
- For pattern 4 (MedAide) and any other domain-specific results, clearly flag domain limitations and avoid implying broad generalizability beyond those domains unless supported.

7) Tighten the overall synthesis narrative
- Consider a brief, explicit opening paragraph that ties patterns to common reliability goals (observability, determinism vs nondeterminism management, governance, latency-reliability tradeoffs) to strengthen flow between sections.

8) Improve the reference map clarity
- Ensure every source that appears in the “References”/“Sources” list is consistently cited in the text and that the arXiv IDs, URLs, and venue/year formatting are uniform.

9) Minor editorial consistency
- Check for minor phrasing and typographical consistency (e.g., “LLM” vs “LLMs” and capitalization of patterns) to improve readability.

## Citation Issues

- Unsupported or overstated claims: Some results are domain-specific or preprint-only; ensure caveats are explicit and consistently placed with the claims.
- Near-claim citations: Confirm that every substantive claim (definitions, results, and representative notes) has a matching in-text citation immediately after it.
- Suspicious or missing citations: The claim of a “clinical trial–adjacent case study” should be cross-checked against the source to ensure proper wording. If needed, adjust to reflect the exact nature of the evidence.
- Broken/logical links: No explicit broken links detected, but ensure every pattern’s “Definition” and “Representative notes” are tied to the cited source in the same sentence or immediately after it.

## Optional Improvements

- Consider a compact, one-page taxonomy diagram or table mapping each pattern to its evidence type, typical use case, and key caveats.
- Add a short “Open research agenda” box listing prioritized next steps (e.g., cross-domain replication, standardization efforts, safety-integration patterns, evaluation benchmarks).
- Include a concise assessment of how these patterns interact with external tools (tool use, external verifiers, and safety layers) as a distinct consideration.

## 4.5 Plan Execution

`execute_plan` is ordinary deterministic workflow code, not an agent. It validates
the plan, dispatches each step to the assigned agent, and records completed artifacts.
The learning version fails fast: an agent exception stops the workflow immediately.

In [45]:
STEP_HANDLERS = {
    "research_agent": research_agent,
    "writer_agent": writer_agent,
    "editor_agent": editor_agent,
}


def execute_plan(
    user_request: str,
    plan: TaskPlan,
    model: str = MODEL_NAME,
    writer_requirements: dict[str, int] | None = None,
    verbose: bool = True,
) -> list[dict[str, Any]]:
    """Run each validated plan step in order and return its artifacts."""

    validate_task_plan(plan, max_steps=len(plan.steps))

    history: list[dict[str, Any]] = []

    for position, step in enumerate(plan.steps, start=1):
        if verbose:
            print(f"[WORKFLOW] Step {position}/{len(plan.steps)} -> {step.agent}")

        extra_arguments = (
            {"report_requirements": writer_requirements}
            if step.agent == "writer_agent"
            else {}
        )
        result = STEP_HANDLERS[step.agent](
            user_request=user_request,
            step=step,
            history=history,
            model=model,
            verbose=verbose,
            **extra_arguments,
        )

        history.append(
            {
                "step_id": step.id,
                "agent": step.agent,
                "task": step.task,
                "result": result,
            }
        )

    if verbose:
        print("[WORKFLOW] Plan completed.")

    return history

In [46]:
def validate_final_report(
    report: str,
    min_words: int = 1200,
    max_words: int = 1800,
) -> int:
    """Check basic output constraints, not factual correctness."""

    if min_words < 1 or max_words < min_words:
        raise ValueError("Report word limits are invalid.")

    report = report.strip()

    if not report:
        raise RuntimeError("Final report is empty.")

    if re.search(r"(?m)^#\s+\S", report) is None:
        raise RuntimeError("Final report must contain a top-level Markdown title.")

    if re.search(r"(?im)^##\s+(references|sources)\s*$", report) is None:
        raise RuntimeError("Final report must contain a References or Sources section.")

    if re.search(r"https?://", report) is None:
        raise RuntimeError("Final report must contain source URLs.")

    word_count = len(report.split())

    if not min_words <= word_count <= max_words:
        raise RuntimeError(
            f"Final report has {word_count} words; expected "
            f"{min_words} to {max_words}."
        )

    return word_count


def run_deep_research(
    topic: str,
    model: str = MODEL_NAME,
    max_steps: int = 6,
    min_report_words: int = 1200,
    max_report_words: int = 1800,
    verbose: bool = True,
) -> str:
    """Plan, execute, validate, and return one deep research report."""

    if min_report_words < 1 or max_report_words < min_report_words:
        raise ValueError("Report word limits are invalid.")

    writer_requirements = {
        "min_words": min_report_words,
        "max_words": max_report_words,
    }

    plan = planner_agent(
        topic=topic,
        model=model,
        max_steps=max_steps,
        verbose=verbose,
    )
    history = execute_plan(
        user_request=topic,
        plan=plan,
        model=model,
        writer_requirements=writer_requirements,
        verbose=verbose,
    )

    final_report = history[-1]["result"]
    word_count = validate_final_report(
        final_report,
        min_words=min_report_words,
        max_words=max_report_words,
    )

    if verbose:
        print(f"[DEEP_RESEARCH] Final report validated: words={word_count}")
        print("[DEEP_RESEARCH] Final report retained in notebook memory.")

    return final_report

In [47]:
final_report = run_deep_research(
    "What are the recent developments in AI design patterns for improving the "
    "reliability of LLM agents?"
)

[PLANNER_AGENT] Starting planning.
[PLANNER_AGENT] Topic: What are the recent developments in AI design patterns for improving the reliability of LLM agents?
[PLANNER_AGENT] Model call #1
[PLANNER_AGENT] Planning completed.
[PLANNER_AGENT] Total steps: 4
[PLANNER_AGENT] Agent route: research_agent -> writer_agent -> editor_agent -> writer_agent
[WORKFLOW] Step 1/4 -> research_agent
[RESEARCH_AGENT] Starting step 1: Research objective: Identify and collect recent design patterns (approx. last 3–5 years) that improve the reliability of LLM agents. Evidence areas to cover: (a) taxonomy of design patterns for reliability (e.g., guardrails, input sanitization, verification, prompt design patterns, tool-use safeguards, fallback/retry strategies), (b) reliability-focused orchestration and coordination patterns (multi-agent collaboration, error handling, retries, monitoring, rollbacks), (c) evaluation and testing patterns (robustness metrics, failure-mode analysis, test harnesses), (d) deploym

In [48]:
Markdown(final_report)

# Reliability-Focused Design Patterns for LLM Agents: A Synthesis of Recent Developments (2021–2026)

## Abstract
This report synthesizes recent design patterns aimed at improving the reliability of LLM-enabled agents, focusing on work from roughly 2021–2026. It outlines a taxonomy of patterns spanning guardrails and access control, data and prompt handling, reasoning and coordination, and evaluation and deployment. The document highlights concrete mechanisms (e.g., layered guardrails, tool-use constraints, structured prompts, and failure-aware learning) and maps them to representative use-cases, empirical evidence, and limitations. It also discusses reliability-oriented orchestration, evaluation practices, and deployment considerations, including real-world case studies and industry guidance. Evidence varies in rigor across sources, with notable gains reported for hierarchical multi-agent coordination and defense against prompt injection, yet standardized, cross-domain benchmarks and long-term deployment data remain gaps. Citations to Step 1 findings are provided to enable verification and future updates.

## Executive Summary
The reliability of LLM agents benefits from a multi-layered design approach that combines architectural guardrails, disciplined data and prompt handling, and structured coordination among reasoning agents. Across the Step 1 results, three patterns stand out for their practical impact: (i) layered guardrails and architectural constraints on tool use, which provide defense-in-depth beyond model behavior; (ii) hierarchical multi-agent coordination, which delivers significant efficiency and robustness gains in complex tasks; and (iii) defense-oriented prompts and input handling, including preference-optimization and structured queries, that reduce susceptibility to prompt injection and data-leakage. Where evaluated, these patterns show meaningful improvements in reliability metrics, such as reduced attack success rates and substantial reductions in simulated engineering costs, but evidence remains domain-specific and limited by standardized cross-domain benchmarks. Real-world deployment experiences underscore the need for ongoing monitoring, human-in-the-loop pathways, and adaptable guardrails to respond to evolving threat vectors. The literature also emphasizes that while these patterns collectively improve reliability, no single pattern suffices; integration and maintenance across evolving toolchains and models are essential. For practitioners, a pragmatic path recommends adopting a four-pillar taxonomy—Guardrails & Access Control; Data & Prompt Handling; Reasoning & Coordination; Evaluation & Deployment—and prioritizing patterns that align with domain risk tolerances and resource constraints. The Step 1 notes identify concrete patterns, their use-cases, and evidence, with explicit citations to arXiv papers and industry guidance to support future validation and standardization efforts.

## Introduction
Recent work has focused on formalizing design patterns that bolster the reliability of LLM agents operating in real-world settings. The Step 1 notes catalog a wide range of approaches, from guardrails and tool-use restrictions to structured prompting and failure-aware learning, complemented by evaluation frameworks and deployment experiences. While several patterns report quantifiable benefits in controlled settings, cross-domain reliability benchmarks and long-term deployment data remain scarce. This report presents a structured taxonomy of the most salient patterns, analyzes how they interrelate through orchestration and evaluation, and discusses practical deployment considerations and limitations, with citations linked to the underlying Step 1 evidence.

## Taxonomy of Reliability Design Patterns
The patterns are organized into four pillars to reflect distinct reliability objectives and engineering implications. Each pattern includes: problem addressed, core mechanism, concrete use-case, evidence/metrics, limitations, and citations.

### Pillar A: Guardrails & Access Control
- Pattern 1: Layered guardrails architecture for LLM agents
  - Problem addressed: Agents can perform unsafe actions, drift over time, or be manipulated by prompts; reliability degrades under real-world use.
  - Core mechanism: A layered safeguards stack—input grounding/validation, least-privilege tool access, output checks, human-in-the-loop, runtime monitoring—implemented alongside the agent.
  - Concrete use-case: Production tool-assisted tasks with safety constraints and ongoing review.
  - Evidence/metrics: Design patterns emphasize multiple safeguards and architectural constraints; industry discussions summarize layered guardrails in production contexts.
  - Limitations: Substantial engineering effort; effectiveness varies by domain; not a universal fix.
  - Citations: http://arxiv.org/abs/2506.08837v2; https://www.nalashaa.com/llm-guardrails-reliable-ai-agents; https://www.kalviumlabs.ai/blog/guardrails-for-llm-applications

- Pattern 2: Architectural constraints on tool use and permissioning
  - Problem addressed: Broad tool access can cause unintended side effects and vulnerabilities.
  - Core mechanism: Enforce architectural constraints that limit tool calls and conditions; sandboxing and least-privilege access.
  - Concrete use-case: Agents with restricted browsing/file access to reduce operational risk.
  - Evidence/metrics/limitations: Emphasized in design-pattern discussions; may constrain capability if not carefully mapped.
  - Citations: http://arxiv.org/abs/2506.08837v2; https://labs.reversec.com/posts/2025/08/design-patterns-to-secure-llm-agents-in-action

- Pattern 3: Input sanitization and least-privilege data handling
  - Problem addressed: Untrusted input can trigger unsafe or unintended actions.
  - Core mechanism: Sanitize inputs prior to ingestion; enforce least-privilege data handling for critical steps.
  - Concrete use-case: Guarded processing of user data and external content.
  - Evidence/metrics/limitations: Security-focused pattern discussions; sanitization rules must adapt to evolving prompt techniques.
  - Citations: http://arxiv.org/abs/2506.08837v2; http://arxiv.org/abs/2402.06363v2; http://arxiv.org/abs/2410.05451v3

### Pillar B: Data & Prompt Handling
- Pattern 4: Structured prompts / separate prompts and data (Structured Queries)
  - Problem addressed: Prompt injections where data streams contaminate or override instructions.
  - Core mechanism: Front-end separation of prompts and data, using structured queries; prompts resilient to data leakage.
  - Concrete use-case: Web/app data interleaved with prompts; secure input handling.
  - Evidence/metrics/limitations: StruQ demonstrates resistance gains; architectural complexity and model tuning needs.
  - Citations: http://arxiv.org/abs/2402.06363v2; https://arxiv.org/abs/2402.06363v2; http://arxiv.org/abs/2410.05451v3

- Pattern 5: Preference-optimization defenses against prompt injection
  - Problem addressed: Prompt injection threats compromising outputs.
  - Core mechanism: Train models to prefer secure outputs using a curated preference dataset.
  - Concrete use-case: Production agents exposed to external data.
  - Evidence/metrics/limitations: SecAlign reports reduced attack success; requires adversarial data curation and retraining.
  - Citations: http://arxiv.org/abs/2410.05451v3

- Pattern 6: Learning from failure trajectories (failure-aware training)
  - Problem addressed: Failures reveal actionable failure modes; ignoring them wastes learning opportunities.
  - Core mechanism: Incorporate failed trajectories with explicit indicators into training to improve error handling and tool use.
  - Concrete use-case: Tool-enabled reasoning tasks; improved performance on reasoning and planning by learning from missteps.
  - Evidence/metrics/limitations: Reported performance gains; data curation challenges to avoid overfitting or unwanted behaviors.
  - Citations: http://arxiv.org/abs/2402.11651v2; https://arxiv.org/pdf/2402.11651v2

### Pillar C: Reasoning & Coordination
- Pattern 7: Hierarchical multi-agent reasoning and coordination for reliability
  - Problem addressed: Single-agent planning can be brittle or resource-intensive for complex tasks.
  - Core mechanism: A hierarchy of agents with defined roles (peer review, triage-ranking, triage-forms) coordinated by higher-level agents.
  - Concrete use-case: Autonomous functional materials discovery; reduced computational costs and interpretable reasoning trajectories.
  - Evidence/metrics/limitations: MASTER framework reports substantial efficiency gains; coordination overhead and generalization considerations exist.
  - Citations: http://arxiv.org/abs/2512.13930v1; https://arxiv.org/pdf/2512.13930v1

- Pattern 8: Evaluation, robustness testing, and test-harness practices
  - Problem addressed: Need reliable metrics, failure-mode analysis, and robust evaluation.
  - Core mechanism: Multi-agent evaluations, adversarial testing, standardized benchmarks, and test harnesses for systematic failure analysis.
  - Concrete use-case: Cross-model evaluation and robustness assessment against adversarial inputs.
  - Evidence/metrics/limitations: Frameworks for cross-model evaluation; adversarial/robustness tasks highlight testing and defense needs; benchmark heterogeneity remains a challenge.
  - Citations: http://arxiv.org/abs/2404.01023v1; https://arxiv.org/pdf/2404.01023v1; http://arxiv.org/abs/2309.11285v1; https://arxiv.org/pdf/2309.11285v1; http://arxiv.org/abs/2505.11717v4; https://arxiv.org/pdf/2505.11717v4

### Pillar D: Evaluation, Deployment, and Real-World Experience
- Pattern 9: Real-world deployment experiences and case studies
  - Problem addressed: Bridging research prototypes and production reliability.
  - Core mechanism: Deployment guidelines, guardrails, monitoring, and iterative improvements documented in industry contexts.
  - Concrete use-case: Production-grade guardrails; practical deployment learnings.
  - Evidence/metrics/limitations: Industry-led accounts provide practical lessons but may lack controlled rigor; generalizability varies by domain.
  - Citations: https://www.nalashaa.com/llm-guardrails-reliable-ai-agents; https://www.kalviumlabs.ai/blog/guardrails-for-llm-applications; https://labs.reversec.com/posts/2025/08/design-patterns-to-secure-llm-agents-in-action

- Pattern 10: Adversarial attacks and defense in LLM-enabled agents
  - Problem addressed: Prompt injection and adversarial data manipulation.
  - Core mechanism: Defenses such as structured patterns (StruQ), preference optimization (SecAlign), and detection/classification approaches.
  - Concrete use-case: Secure web and multi-modal agents; defense against evolving attack vectors.
  - Evidence/metrics/limitations: Attack-defense dynamics are an ongoing arms race; defenses require ongoing adaptation.
  - Citations: http://arxiv.org/abs/2505.11717v4; http://arxiv.org/abs/2402.06363v2; http://arxiv.org/abs/2410.05451v3

## Reliability-Oriented Orchestration Patterns
Reliability improves when orchestration patterns coordinate multiple reasoning agents, manage failures, and provide structured decision flows. Key insights include:
- Hierarchical coordination (Pattern 7) enables defined roles (peer review, triage) that improve problem-solving efficiency and provide interpretable reasoning trajectories, albeit with coordination overhead and uncertainties about broad generalization. [http://arxiv.org/abs/2512.13930v1]
- Error handling and retry mechanisms, anchored in runtime monitoring, support resilient operation and dynamic task reallocation. These align with layered guardrails (Pattern 1) and restricted tool-use patterns (Pattern 2). [http://arxiv.org/abs/2506.08837v2]
- Monitoring and human-in-the-loop pathways are integral to production safety and post-hoc review, complementing automated safeguards. [http://arxiv.org/abs/2506.08837v2]
- Failure-aware training (Pattern 6) informs orchestration by shaping how agents respond to missteps, improving recoverability across tool-use and reasoning steps. [http://arxiv.org/abs/2402.11651v2]

Implications for practitioners:
- Design orchestration to balance autonomy and oversight; prefer hierarchical delegation for complex/long-horizon tasks.
- Integrate monitoring and HITL as core infrastructure rather than add-ons.
- Align failure-trajectory learning with orchestration policies to improve recoverability and trust.

## Evaluation, Testing Approaches
A robust evaluation regime is central to establishing reliability:
- Cross-model evaluation frameworks enable systematic comparisons across models and tasks. [http://arxiv.org/abs/2404.01023v1]
- Adversarial testing and robustness tasks quantify resilience to prompt injections and related threats; defenses such as StruQ and SecAlign illustrate concrete approaches. [http://arxiv.org/abs/2402.06363v2; http://arxiv.org/abs/2410.05451v3; http://arxiv.org/abs/2505.11717v4]
- Standardized test harnesses and failure-mode analyses reveal reliability bottlenecks beyond success metrics. [http://arxiv.org/abs/2404.01023v1]
- Real-world deployment studies provide pragmatic validation, though their generalizability is domain-dependent. [https://www.nalashaa.com/llm-guardrails-reliable-ai-agents; https://www.kalviumlabs.ai/blog/guardrails-for-llm-applications]

Note on evidence quality:
- Strong, quantifiable gains are reported for hierarchical coordination and certain defense mechanisms, but results are often domain-specific and vary with datasets and tasks. Gaps remain in standardized cross-domain benchmarks and long-term deployment outcomes.

## Practical Considerations, Deployment Implications, and Limitations
- Deployment scales and monitoring require integration into production toolchains, with clear escalation and rollback pathways. Guardrails and monitoring are essential, but domain-specific adaptation is necessary. [Pattern 9 research notes]
- Engineering effort and integration costs are nontrivial for layered guardrails (Pattern 1) and tool-use constraints (Pattern 2). Effectiveness is context-dependent. [Pattern 1, Pattern 2]
- Data handling patterns (Pattern 3) and structured prompting (Pattern 4) demand robust front-end design and potentially model tuning, with ongoing adaptation to new prompt techniques. [Pattern 3, Pattern 4]
- Adversarial adaptation is ongoing; defenses like preference optimization (Pattern 5) and StruQ/SecAlign (Pattern 10) require continual data curation and retraining. [Pattern 5, Pattern 10]
- Standardization gaps: Cross-domain reliability metrics and benchmarks remain underdeveloped, challenging broad generalization. [Pattern 8; Notes on evidence gaps]
- Real-world accounts provide practical guidance but may lack the rigor of controlled studies; results must be interpreted with domain context in mind. [Pattern 9]

## References
- http://arxiv.org/abs/2506.08837v2
- https://arxiv.org/pdf/2506.08837v2
- https://www.nalashaa.com/llm-guardrails-reliable-ai-agents
- https://www.kalviumlabs.ai/blog/guardrails-for-llm-applications
- https://labs.reversec.com/posts/2025/08/design-patterns-to-secure-llm-agents-in-action
- http://arxiv.org/abs/2402.06363v2
- https://arxiv.org/pdf/2402.06363v2
- http://arxiv.org/abs/2410.05451v3
- https://arxiv.org/pdf/2410.05451v3
- http://arxiv.org/abs/2402.11651v2
- https://arxiv.org/pdf/2402.11651v2
- http://arxiv.org/abs/2512.13930v1
- https://arxiv.org/pdf/2512.13930v1
- http://arxiv.org/abs/2501.05032v2
- https://arxiv.org/pdf/2501.05032v2
- http://arxiv.org/abs/2404.01023v1
- https://arxiv.org/pdf/2404.01023v1
- http://arxiv.org/abs/2309.11285v1
- https://arxiv.org/pdf/2309.11285v1
- http://arxiv.org/abs/2505.11717v4
- https://arxiv.org/pdf/2505.11717v4

Notes on sources: The references above are deduplicated to the extent possible and preserve the original URLs exactly as provided in the Step 1 notes. Citations in the narrative correspond to explicit Step 1 findings and aims to enable future validation and updates.

# 5. Notebook-Contained Report

Generated reports remain in memory and are rendered directly by the Notebook.
This learning version does not create a separate Markdown report file. Keeping the
code, execution log, and final artifact together makes each version self-contained.
